# Numerai Classic v6.0 ("Unity") — Deterministic Baselines A, E & Persistence (`NumeraiExtractions` + `BinSnappingEstimator`, `LagReversalEstimator` & `PersistenceEstimator`)

- **Data Loading (`data_utils.load_unity_numerai_data`)**: Resolves and downloads the Unity v6.0 Parquets using the official Kaggle clients (`kagglehub.dataset_download` and `kaggle.api.kaggle_api_extended.KaggleApi`) with `numerapi.NumerAPI` fallback, catching each library's actual exception classes (`KaggleApiHTTPError`, `UnauthenticatedError`, `CredentialError`, `NotFoundError`, `DataCorruptionError`, `BackendError`, `KaggleEnvironmentError`, `ColabEnvironmentError`, `kaggle.rest.ApiException`, `HTTPError`, `OSError`, `ValueError`).
- **Exploratory Data Analysis (`# EDA`)**: Lists available `TARGET_COL` and `feature_set` options from `features.json`, inspects `train_df.head(2)` / `validation_df.head(2)` and `info()`, plots era sizes over time (`groupby(ERA_COL)[TARGET_COL].size()`), overlays feature standard-deviation distributions across Train vs. Validation, and displays the top 10 features correlated with `TARGET_COL`.

---

### 1. Feature Extraction Parameters (`utils.NumeraiExtractions` — `estimator__extractor__*`)
- **`era_col`** (`[ERA_COL]`): Column name containing the sequential weekly time index (`'era'`).
- **`ticker_col`** (`[TICKER_COL]`): Column name containing the security identifier (`'numerai_ticker'`).
- **`target_col`** (`[TARGET_COL, 'target_cyrusd_20', 'target_teager2b_20']`): Target column used when ranking features by target correlation.
- **`year_horizon_filter`** (`[None, (1, 2, 3), (4, 5, 6), (7, 8, 9)]`): Restricts `fit()` variance and correlation estimation to specific 52-week training year buckets `(era - 1) // 52 + 1` (`None` uses all training years; never exposed as a model feature).
- **`variance_threshold`** (`[None, 0.02, 0.05, 0.10]`): Drops near-constant features whose training variance is $\le$ threshold via `sklearn.feature_selection.VarianceThreshold` (`None` keeps all features).
- **`correlation_method`** (`[None, 'pandas_mean', 'numerai_mean', 'pearson', 'spearman']`):
  - `None`: Skips correlation ranking and keeps original column order.
  - `'pandas_mean'`: Computes per-era Pearson correlation (`era_df.corrwith(era_target)`) and averages across eras.
  - `'numerai_mean'`: Computes official per-era `numerai_tools.scoring.numerai_corr` and averages across eras.
  - `'pearson'`: Global pooled Pearson correlation (`r_regression`) across all rows.
  - `'spearman'`: Global pooled Spearman rank correlation across all rows.
- **`abs_corr`** (`[True, False]`): `True` ranks features by $|\text{corr}|$ (strongest magnitude in either direction); `False` ranks by signed $\text{corr}$ (positive only first).
- **`top_k`** (`[None, 10, 20, 40]`): Keeps only the top $K$ highest-ranked features from the extractor (`None` keeps all features).
- **`feature_selection_percentile`** (`[None, 25.0, 50.0, 75.0, 100.0]`): Keeps the top percentage of ranked features (`None` keeps all features).
- **`sparse_selector`** (`[None, 'lasso', 'elasticnet']`) & **`sparse_alpha`** (`[0.001, 0.01, 0.1]` / `Real(1e-4, 1e-1, prior='log-uniform')`): Optional multivariate sparse feature selection via `sklearn.feature_selection.SelectFromModel` with `Lasso` or `ElasticNet` fitted strictly inside `fit()` (`None` disables).
- **`pca_n_components`** (`[None, 10, 20, 40]` / `0.90`): Optional `sklearn.decomposition.PCA` compression fitted strictly inside `fit()` and quantized to `0..4` `int8` bins in `transform()` to reduce feature exposure without dropping features (`None` disables).
- **`quantize_scope`** (`['era', 'train_fit', 'global']`) & **`quantize_strategy`** (`['zscore', 'quantile']`): Controls how continuous derived features are quantized into `0..4` `int8` bins (`'era'` = leak-free within-era cross-sectional quantization; `'train_fit'` = frozen `StandardScaler` fitted strictly on training data in `fit()`; `'global'` = legacy full-array quantization; `'zscore'` = z-score rounding vs. `'quantile'` = `KBinsDiscretizer` quantile edges).
- **`sequence_top_n`** (`[4, 2, 8]`): Number of top-ranked features expanded by rolling-era, lag-diff, and per-ticker sequence extractors.
- **`missing_fill_value`** (`[2.0, 0.0, 4.0, 'mean', 'median', 'era_mean', 'era_median']`):
  - `2.0`, `0.0`, `4.0`, `'mean'`, `'median'`: Imputes missing values via `sklearn.impute.SimpleImputer` (`2.0` is the neutral middle bin of Numerai's `0..4` scale).
  - `'era_mean'` / `'era_median'`: Fills missing values with each feature's within-era cross-sectional mean or median.
- **`feature_groups`** (`unity_feature_groups` on `NumeraiExtractions`, `None` if no thematic groups exist in `feature_metadata['feature_sets']`; omitted from CV search spaces to prevent console log bloat) & **`basic_group_stats`** (`[None, 'mean', 'std', 'skew', 'mean|std', 'mean|skew', 'std|skew', 'mean|std|skew']`): Computes leak-free row-wise summary statistics (`mean`, `std`, `skew` quantized to `0..4` bins) across real Unity v6.0 feature groups or built-in thematic prefixes (`None` disables).
- **`fourier_period_eras`** (`[52, 26, 13]`): Cycle length in eras (52 = annual, 26 = semi-annual, 13 = quarterly) used by `fourier_order`.
- **`fourier_order`** (`[None, 1, 2, 3]`): Sole cyclical seasonality encoding — adds smooth $\sin(2\pi k \cdot \text{era} / P)$ and $\cos(2\pi k \cdot \text{era} / P)$ harmonics for $k=1..\text{fourier\_order}$ quantized to `0..4` bins (`None` disables).
- **`include_rolling_era`** (`[False, True]`) & **`rolling_era_windows`** (`[(4, 12), (2, 6, 12), (4, 8, 16)]`): When `include_rolling_era=True`, computes causal rolling era means over each window length $W$ in `rolling_era_windows` for the top correlated features, quantized to `0..4` bins.
- **`include_lag_diff`** (`[False, True]`) & **`era_lag_scales`** (`[(1, 2, 4), (1, 2, 4, 8), (2, 4, 8)]`): When `include_lag_diff=True`, computes causal era-to-era lag differences ($\bar{x}_e - \bar{x}_{e-k}$) for each lag step $k$ in `era_lag_scales`, quantized to `0..4` bins.
- **`ticker_features`** (`[None, 'rolling_mean', 'momentum_diff', 'weeks_since_high', 'rolling_rank', 'weeks_since_change', 'expanding_stats', 'raw_lags', 'rolling_mean|momentum_diff', 'rolling_mean|momentum_diff|rolling_rank', 'expanding_stats|raw_lags', 'rolling_mean|momentum_diff|weeks_since_high|rolling_rank|weeks_since_change|expanding_stats|raw_lags']`), **`ticker_windows`** (`[(4, 12), (2, 6, 12)]`), & **`ticker_lag_steps`** (`[None, (1, 2), (1, 2, 4)]`): Computes causal per-ticker time-series features (`rolling_mean`, `momentum_diff`, vectorized `weeks_since_high`, `rolling_rank`, vectorized `weeks_since_change`, `expanding_stats` [`expanding().mean()/min()/max()`], and `raw_lags` [`shift(k)` for $k \in$ `ticker_lag_steps`]) grouped strictly by `numerai_ticker` sorted chronologically by `era` (`None` disables).
- **`era_dispersion_features`** (`[None, 'mean', 'std', 'extreme_frac', 'mean|std', 'mean|extreme_frac', 'std|extreme_frac', 'mean|std|extreme_frac']`): Adds per-row cross-sectional mean (`'mean'`), standard deviation (`'std'`), and fraction of extreme `0`/`4` bins (`'extreme_frac'`) across selected features (`None` disables).

---

### 2. Shared Parent Estimator Parameters (`baseline_utils.BaseNumeraiEstimator` — `estimator__*`)
- **`correlation_method`** (`[None, 'pandas_mean', 'numerai_mean', 'pearson', 'spearman']`): How the estimator scores and weights post-extraction features against `TARGET_COL`.
- **`abs_corr`** (`[True, False]`): `True` selects top features by $|\text{corr}|$; `False` selects by signed $\text{corr}$.
- **`top_k`** (`[None, 10, 20, 40]`): Number of top post-extraction features used by the estimator (`None` uses all extracted features).
- **`era_weighting`** (`[None, 'equal_era', 'recency_linear', 'recency_exp']`):
  - `None`: Equal weight per row.
  - `'equal_era'`: Equal total weight per era ($1 / N_{\text{era}}$) so large eras do not dominate.
  - `'recency_linear'`: Linearly increases era weight from oldest ($0.5$) to newest ($1.5$).
  - `'recency_exp'`: Exponentially decays older eras by $\exp(-\text{recency\_decay} \cdot (e_{\max} - e))$.
- **`recency_decay`** (`Real(0.001, 0.020)`): Exponential decay rate per era when `era_weighting='recency_exp'`.
- **`prediction_transform`** (`[None, 'era_zscore', 'era_rank']`):
  - `None`: Returns raw model predictions.
  - `'era_zscore'`: Standardizes predictions to zero mean and unit variance within each era.
  - `'era_rank'`: Converts predictions to `[0, 1]` percentile ranks within each era.
- **`bias_correction`** (`[None, 'mean_residual']`): When `'mean_residual'`, records `mean(y_train - y_hat_train)` in `fit()` and adds it to predictions in `predict()` (`None` disables).

---

### 3. Child Baseline Estimator Parameters (`BinSnappingEstimator`, `LagReversalEstimator` & `PersistenceEstimator`)
- **Baseline A — `BinSnappingEstimator` (Map Matching Lookup Table)**:
  - **`n_bins`** (`[5, 3, 7]`): Number of discrete integer bins per feature (`5` matches Numerai's `0..4` bins).
  - **`smoothing`** (`[0.0, 25.0, 50.0, 100.0]` / `Real(0.0, 100.0)`): Empirical-Bayes shrinkage weight pulling sparse bin averages toward the global target mean ($\frac{\sum y + m \cdot \bar{y}}{N + m}$).
  - **`agg_mode`** (`['mean', 'corr_weighted']`): `'mean'` averages snapped bin predictions equally across selected features; `'corr_weighted'` weights each feature's bin lookup by $|\text{corr}_j|$.
- **Baseline E — `LagReversalEstimator` (Era Autocorrelation / PACF Sign & Shrinkage Rule)**:
  - **`lag`** (`[1, 2, 4, 6]` / `Integer(1, 6)`): Number of era steps $k$ used to compute each feature's era-level lag-$k$ serial dependence.
  - **`acf_mode`** (`['feature_mean', 'feature_corr']`): `'feature_mean'` computes lag-$k$ dependence on each feature's per-era mean series; `'feature_corr'` computes lag-$k$ dependence on each feature's per-era target-correlation series.
  - **`acf_metric`** (`['acf', 'pacf']`): `'acf'` uses lag-$k$ autocorrelation; `'pacf'` uses lag-$k$ partial autocorrelation (`statsmodels.tsa.stattools.pacf`).
  - **`acf_detrend`** (`[None, 'linear', 'constant']`): Detrends the era series via `scipy.signal.detrend` before computing $\text{ACF}(k)$/$	ext{PACF}(k)$ (`None` = raw series, `'linear'` = removes linear time trend, `'constant'` = subtracts series mean).
  - **`acf_threshold`** (`[-0.05, 0.0, 0.05]` / `Real(-0.20, 0.20)`): Cutoff threshold $\tau$ for sign inversion or continuous shrinkage.
  - **`sign_mode`** (`['hard', 'soft_threshold', 'tanh']`): `'hard'` applies $\pm 1$ sign flip around `acf_threshold`; `'soft_threshold'` applies continuous soft-thresholded shrinkage; `'tanh'` applies smooth $\tanh((\text{acf} - \tau)/0.05)$ shrinkage.
  - **`center_features`** (`[True, False]`): `True` centers each feature by its training mean before applying the signed correlation weights; `False` multiplies raw bin values.
- **Brownlee Persistence Baseline — `PersistenceEstimator` (Decoupled Per-Ticker Target Persistence / Zero-Rule)**:
  - **`ticker_col`** (`[TICKER_COL]`): Security identifier column (`'numerai_ticker'`).
  - **`persistence_mode`** (`['last', 'expanding_mean', None]`): `'last'` predicts each ticker's most recent settled training target; `'expanding_mean'` predicts each ticker's historical training target mean; `None` falls back to Brownlee's Zero-Rule baseline (global training target mean).
  - **`persistence_lag`** (`[4, 1, 2, 8, None]`): Minimum era lag required before a training target is treated as settled (`4` for 20-day targets; `None` uses all available training history).
  - **`smoothing`** (`[0.0, 10.0, 25.0, 50.0]` / `Real(0.0, 100.0)`): Shrinkage pulling ticker target history toward the global training mean (`0.0` or `None` = no shrinkage).

---

- **Evaluation (`utils.evaluate_numerai_predictions`)**: Reports official `numerai_tools.scoring.numerai_corr`, per-era `Spearman Correlation`, `Average Payout` (`(corrs / 0.2).clip(-1, 1)`), `Sharpe Ratio` (`corrs.mean() / corrs.std()`), `Mean Absolute Error (MAE)`, `Forecast Bias` (`mean(y_true - y_pred)`), `Directional Hit Rate` (sign agreement vs. `0.5`), `Era CORR Lag-1 ACF`, `Ljung-Box p-value` (`acorr_ljungbox`), `ARCH-LM p-value` (`het_arch`), `ADF Stationarity p-value` (`adfuller`), `Skewness`, `Excess Kurtosis`, `Jarque-Bera p-value`, `Max Drawdown`, and validation `Feature Exposure`.

In [1]:
import gc
import os
import sys
import json
import time
import warnings
import importlib
from pathlib import Path
from datetime import datetime

python_env_dir = Path(sys.executable).parent
library_bin_dir = python_env_dir / 'Library' / 'bin'
if library_bin_dir.is_dir() and str(library_bin_dir) not in os.environ.get('PATH', ''):
    os.environ['PATH'] = str(library_bin_dir) + os.pathsep + str(python_env_dir) + os.pathsep + os.environ.get('PATH', '')

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
try:
    import seaborn as sns
except ImportError:
    sns = None
from IPython.display import display

warnings.filterwarnings('ignore')

# Local, Kaggle (/kaggle/input, /kaggle/usr/lib, /kaggle/working), & nested dataset path routing
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir).lower() == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)
if os.path.isdir(src_path) and src_path not in sys.path:
    sys.path.insert(0, src_path)

module_search_roots = [
    Path(workspace_root),
    Path('/kaggle/working'),
    Path('/kaggle/usr/lib'),
    Path('/kaggle/input'),
]
for search_root in module_search_roots:
    if not search_root.is_dir():
        continue
    for target_file in ('afml_utils.py', 'data_utils.py', 'utils.py', 'baseline_utils.py'):
        for found_path in search_root.rglob(target_file):
            parent_dir = str(found_path.parent)
            if parent_dir not in sys.path:
                sys.path.insert(0, parent_dir)

from pandas.plotting import autocorrelation_plot
from scipy.stats import spearmanr
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from statsmodels.graphics.gofplots import qqplot

from numerai_tools.scoring import numerai_corr

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

try:
    import wandb
except ImportError:
    wandb = None

try:
    import afml_utils
    import data_utils
    import utils
    import baseline_utils
except ModuleNotFoundError as import_err:
    mounted_kaggle_inputs = [str(p) for p in sorted(Path('/kaggle/input').iterdir())] if Path('/kaggle/input').is_dir() else []
    raise ModuleNotFoundError(
        f"{import_err}. Could not locate data_utils.py / utils.py / baseline_utils.py in "
        f"workspace_root={workspace_root}, /kaggle/working, /kaggle/usr/lib, or /kaggle/input "
        f"(mounted inputs: {mounted_kaggle_inputs}). "
        "On Kaggle, attach a Dataset or Utility Script containing src/data_utils.py, src/utils.py, and src/baseline_utils.py."
    ) from import_err
importlib.reload(afml_utils)
importlib.reload(data_utils)
importlib.reload(utils)
importlib.reload(baseline_utils)
from data_utils import (
    compute_per_ticker_timeseries_features,
    discover_and_fetch_v6_meta_or_benchmark,
    load_unity_numerai_data,
    load_unity_numerai_data_cached,
)
from utils import (
    NumeraiExtractions,
    build_experiment_setup,
    build_conditional_bayesian_spaces,
    build_quick_results_df,
    candidate_score,
    compute_r1399_churn_retention,
    compute_versioned_candidate_score,
    make_candidate_scorer,
    compute_eda_era_stationarity,
    compute_era_error_envelope,
    plot_error_bin_waterfall,
    configure_error_binning,
    compute_model_error_bin_comparison,
    compute_error_bin_waterfall,
    compute_feature_exposure,
    compute_feature_target_correlations,
    compute_per_era_mmc,
    compute_per_era_numerai_corr,
    compute_per_era_spearman_corr,
    deduplicate_param_grid,
    drop_constant_dims,
    evaluate_numerai_predictions,
    export_diagnostics_predictions,
    plot_era_error_envelope,
    plot_stake_pnl_dashboard,
    plot_stake_pnl_monte_carlo,
    prepare_bayesian_space,
    quantize_to_5bins,
    reset_cv_metric_log,
    run_numereval_diagnostics,
    sharpe_ratio,
    build_live_staking_scenarios_df,
    compute_deflated_sharpe_ratio,
    compute_probabilistic_sharpe_ratio,
    simulate_live_staking_scenarios,
    simulate_stake_pnl,
    simulate_stake_pnl_monte_carlo,
    walk_forward_predict,
    CombinatorialPurgedEraCV,
    PurgedEraKFold,
    build_era_cv_splits,
    compute_clustered_mda,
    compute_drawdown_and_tuw,
    compute_hhi_concentration,
    compute_lo_adjusted_sharpe,
    compute_min_track_record_length,
    compute_pbo_cscv,
    compute_per_era_fnc,
    compute_strategy_risk,
    neutralize_predictions,
    purge_train_before_validation,
    summarize_afml_backtest_diagnostics,
)
from baseline_utils import (
    BaseNumeraiEstimator,
    BinSnappingEstimator,
    LagReversalEstimator,
    PersistenceEstimator,
    configure_extraction_cache,
    run_baseline,
)

print('Imports loaded successfully')
print(f'workspace_root={workspace_root} | SKOPT_AVAILABLE={SKOPT_AVAILABLE}')

Imports loaded successfully
workspace_root=C:\Users\maran\OneDrive\Documents\Git Profile\Data-Projects\numerai | SKOPT_AVAILABLE=True


# Setup

In [2]:
# --- Experiment Knobs (only things you actually change) ---
TARGET_COL = 'target'          # Training target (auto-sets horizon & embargo)
feature_set = 'small'          # 'small', 'medium', or 'all'
era_stride = 4                 # 1, 2, 4 (20d), or 12 (60d)

baseline_mode = 'all'          # 'baseline_a', 'baseline_e', 'persistence', 'both', 'all'
search_mode = 'bayes'          # 'grid' or 'bayes'
score_version = 'v1_skepta'    # 'v0_default', 'v1_skepta', or 'v2_skepta_mmc'
eval_mode = 'both'             # 'static', 'walk_forward', or 'both'
cv_strategy = 'purged_kfold'   # 'timeseries', 'purged_kfold' (AFML Ch. 7), or 'cpcv' (AFML Ch. 12)
n_splits = 5
cv_split = 0.8                 # Chronological CV train/test split ratio (used when n_splits == 1)
downsample_mode = 'within_era'         # None, 'within_era', 'recent', 'earliest', 'uniform', or 'random_eras'
downsample_frac = 0.01          # Fraction of strided training data to keep in (0.0, 1.0]
n_iter = 5
n_jobs = 1                    # Use 1 for feature_set='all' to save RAM
save_results = True
error_bin_mode = 'pct'         # 'pct' (% error + zero-target tol) or 'abs' (raw residual bins)

setup_cfg = build_experiment_setup(
    target_col=TARGET_COL,
    feature_set=feature_set,
    era_stride=era_stride,
    model_tag='baselines_ae',
    baseline_mode=baseline_mode,
    search_mode=search_mode,
    score_version=score_version,
    eval_mode=eval_mode,
    cv_strategy=cv_strategy,
    n_splits=n_splits,
    cv_split=cv_split,
    downsample_mode=downsample_mode,
    downsample_frac=downsample_frac,
    n_iter=n_iter,
    n_jobs=n_jobs,
    save_results=save_results,
    error_bin_mode=error_bin_mode,
    workspace_root=workspace_root,
)
globals().update(setup_cfg)

Experiment: numerai_v6.0_all_eras_stride4_baselines_ae | baseline_mode=all | search_mode=bayes | cv_strategy=timeseries (cv_split=0.8) | ERA_COL=era | TARGET_COL=target | horizon=12 | embargo=3 | downsample=within_era@0.01
[Overlap Notice] TARGET_COL='target' has horizon=12 eras while era_stride=4 (~3.0x label overlap). Consider era_stride=12 or era_weighting='uniqueness'.


# Prep

In [3]:
_prep_t0 = time.perf_counter()
train_df, validation_df, feature_columns, feature_metadata = load_unity_numerai_data_cached(
    data_version=data_version,
    data_folder=data_folder,
    kaggle_dataset=kaggle_dataset,
    feature_set=feature_set,
    target_col=TARGET_COL,
    era_stride=era_stride,
    downsample_mode=downsample_mode,
    downsample_frac=downsample_frac,
    random_state=random_state,
    ticker_timeseries_features=ticker_timeseries_features,
    ticker_rolling_windows=ticker_rolling_windows,
    ticker_lag_steps=ticker_lag_steps,
    workspace_root=workspace_root,
    auto_download=True,
    use_numerapi_fallback=True,
    force_reload=force_reload_data_cache,
)
prep_elapsed_s = time.perf_counter() - _prep_t0
print(f'Loaded {len(feature_columns)} int8 features | train={train_df.shape} ({train_df[ERA_COL].nunique()} eras) | validation={validation_df.shape} ({validation_df[ERA_COL].nunique()} eras)')

Loaded 101 int8 features | train=(10834, 106) (209 eras) | validation=(2660746, 106) (399 eras)


## Exploratory Data Analysis (EDA)

In [4]:
available_targets = list(feature_metadata.get('targets', [TARGET_COL]))
print(f'Available TARGET_COL options ({len(available_targets)}):', available_targets)
print('Available feature_set options:', {k: len(v) for k, v in feature_metadata.get('feature_sets', {}).items()})

print('\nTraining data:')
display(train_df.head(2))
print('Validation data:')
display(validation_df.head(2))
print('Training set info:')
train_df.info()
print('\nValidation set info:')
validation_df.info()

Available TARGET_COL options (24): ['target', 'target_raw_return_20', 'target_raw_return_60', 'target_factor_neutral_20', 'target_factor_neutral_60', 'target_factor_feat_neutral_20', 'target_factor_feat_neutral_60', 'target_camille_20', 'target_camille_60', 'target_sydney_20', 'target_sydney_60', 'target_irina_20', 'target_irina_60', 'target_alexandra_20', 'target_alexandra_60', 'target_colleen_20', 'target_colleen_60', 'target_jean_20', 'target_jean_60', 'target_eleven_4', 'target_eleven_20', 'target_eleven_60', 'target_jupiter_20', 'target_jupiter_60']
Available feature_set options: {'small': 101, 'medium': 351, 'all': 1040, 'signals': 23}

Training data:


,date,data_type,numerai_ticker,feature_intelligence_1,feature_intelligence_41,feature_charisma_2,feature_charisma_5,feature_charisma_7,feature_charisma_11,feature_charisma_13,...,feature_wisdom_67,feature_wisdom_75,feature_wisdom_78,feature_wisdom_96,feature_endurance_7,feature_endurance_13,feature_endurance_69,feature_country,target,era
204,2003-01-03,train,669 HK,3,2,4,0,4,0,2,...,3,2,2,2,2,2,2,2,0.50,1
212,2003-01-03,train,6762 JP,3,2,1,0,3,3,0,...,2,2,3,2,2,2,2,2,0.75,1


Validation data:


,date,data_type,numerai_ticker,feature_intelligence_1,feature_intelligence_41,feature_charisma_2,feature_charisma_5,feature_charisma_7,feature_charisma_11,feature_charisma_13,...,feature_wisdom_67,feature_wisdom_75,feature_wisdom_78,feature_wisdom_96,feature_endurance_7,feature_endurance_13,feature_endurance_69,feature_country,target,era
0,2019-01-04,validation,000030 KR,2,2,4,3,3,1,4,...,0,3,1,3,3,4,1,2,0.5,836
1,2019-01-04,validation,000060 KR,2,2,3,4,4,1,4,...,2,3,3,1,1,3,0,2,0.5,836


Training set info:
<class 'pandas.core.frame.DataFrame'>
Index: 10834 entries, 204 to 1082590
Columns: 106 entries, date to era
dtypes: float32(1), int32(1), int8(101), object(3)
memory usage: 1.5+ MB

Validation set info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2660746 entries, 0 to 2660745
Columns: 106 entries, date to era
dtypes: float32(1), int32(1), int8(101), object(3)
memory usage: 337.5+ MB


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

train_era_sizes = train_df.groupby(ERA_COL)[TARGET_COL].size()
val_era_sizes = validation_df.groupby(ERA_COL)[TARGET_COL].size()
axes[0].plot(train_era_sizes.index, train_era_sizes.to_numpy(), color='#2563eb', linewidth=1.8, label='Train')
axes[0].plot(val_era_sizes.index, val_era_sizes.to_numpy(), color='#10b981', linewidth=1.8, label='Validation')
axes[0].set_title('Era Sizes Over Time', weight='bold', fontsize=13)
axes[0].set_xlabel('Era')
axes[0].set_ylabel('Rows per Era')
axes[0].legend()

train_feat_stds = train_df[feature_columns].astype(np.float32).std()
val_feat_stds = validation_df[feature_columns].astype(np.float32).std()
axes[1].hist(train_feat_stds, bins=30, alpha=0.55, density=True, color='#2563eb', label='Train')
axes[1].hist(val_feat_stds, bins=30, alpha=0.55, density=True, color='#10b981', label='Validation')
axes[1].set_title('Standard Deviations Over All Features', weight='bold', fontsize=13)
axes[1].set_xlabel('Feature Standard Deviation')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

corr_with_target = train_df[feature_columns].corrwith(train_df[TARGET_COL]).abs().sort_values(ascending=False)
print('Top 10 Features according to absolute correlation with target:')
display(corr_with_target.head(10))

eda_stationarity_df = compute_eda_era_stationarity(train_df, list(corr_with_target.head(10).index), era_col=ERA_COL)
print('ADF Stationarity Check on Top 10 Features (Per-Era Means):')
display(eda_stationarity_df)

Top 10 Features according to absolute correlation with target:


feature_constitution_172    0.033100
feature_charisma_77         0.023878
feature_endurance_13        0.023622
feature_charisma_5          0.020802
feature_strength_71         0.019467
feature_charisma_148        0.019466
feature_constitution_227    0.018298
feature_constitution_217    0.017863
feature_constitution_299    0.017192
feature_dexterity_53        0.016499
dtype: float64

ADF Stationarity Check on Top 10 Features (Per-Era Means):


,feature,adf_stat,adf_pvalue,stationary_at_5pct
0,feature_constitution_172,-6.5333,0.0,True
1,feature_charisma_77,-6.4542,0.0,True
2,feature_endurance_13,-5.6047,0.0,True
3,feature_charisma_5,-7.2296,0.0,True
4,feature_strength_71,-5.7958,0.0,True
5,feature_charisma_148,-7.4745,0.0,True
6,feature_constitution_227,-6.2628,0.0,True
7,feature_constitution_217,-5.9358,0.0,True
8,feature_constitution_299,-6.4826,0.0,True
9,feature_dexterity_53,-6.3866,0.0,True


In [6]:
input_columns = [ERA_COL] + ([TICKER_COL] if TICKER_COL in train_df.columns and TICKER_COL in validation_df.columns else []) + feature_columns
X_train = train_df[input_columns].reset_index(drop=True)
y_train = train_df[TARGET_COL].astype(np.float32).reset_index(drop=True)
train_eras = train_df[ERA_COL].astype(np.int32).reset_index(drop=True)

X_validation = validation_df[input_columns].reset_index(drop=True)
y_validation = validation_df[TARGET_COL].astype(np.float32).reset_index(drop=True)
validation_eras = validation_df[ERA_COL].astype(np.int32).reset_index(drop=True)

# Retain only lightweight validation metadata for diagnostics export, date-aware staking, and free raw DataFrames
val_diag_meta = validation_df[[c for c in ('id', 'numerai_ticker', 'date', ERA_COL) if c in validation_df.columns]].reset_index(drop=True).copy()
val_era_dates = (
    val_diag_meta.groupby(ERA_COL, sort=True)['date'].first()
    if ('date' in val_diag_meta.columns and ERA_COL in val_diag_meta.columns)
    else None
)
del train_df, validation_df
gc.collect()

# Era-level purged CV splitter supporting 'timeseries', 'purged_kfold' (AFML Ch. 7), and 'cpcv' (AFML Ch. 12)
cv_object = build_era_cv_splits(
    train_eras,
    cv_strategy=cv_strategy,
    n_splits=n_splits,
    cv_split=cv_split,
    target_horizon_eras=target_horizon_eras,
    embargo_eras=embargo_eras,
)
print(
    f'Built {len(cv_object)} strictly era-purged CV fold(s) using cv_strategy={cv_strategy!r} '
    f'(horizon={target_horizon_eras} eras, embargo={embargo_eras} eras)'
)


In [7]:
raw_feature_sets = feature_metadata.get('feature_sets', {})
meta_set_names = {'small', 'medium', 'all'}
active_feature_set_cols = set(feature_columns)
unity_feature_groups_dict = {
    group_name: tuple(col for col in group_cols if col in active_feature_set_cols)
    for group_name, group_cols in raw_feature_sets.items()
    if group_name not in meta_set_names and isinstance(group_cols, list)
}
unity_feature_groups_dict = {k: v for k, v in unity_feature_groups_dict.items() if len(v) >= 2}
unity_feature_groups = tuple(sorted(unity_feature_groups_dict.items())) if unity_feature_groups_dict else None

numerai_extractions = NumeraiExtractions(
    era_col=ERA_COL,
    target_col=TARGET_COL,
    ticker_col=TICKER_COL,
    year_horizon_filter=None,
    variance_threshold=None,
    correlation_method='pandas_mean',
    abs_corr=True,
    top_k=None,
    feature_selection_percentile=None,
    sparse_selector=None,
    sparse_alpha=0.01,
    pca_n_components=None,
    quantize_scope='era',
    quantize_strategy='zscore',
    sequence_top_n=4,
    feature_groups=unity_feature_groups,
    basic_group_stats=None,
    fourier_period_eras=52,
    fourier_order=None,
    rolling_era_windows=(4, 12),
    include_rolling_era=False,
    era_lag_scales=(1, 2, 4),
    include_lag_diff=False,
    ticker_features=None,
    ticker_windows=(4, 12),
    ticker_lag_steps=None,
    era_dispersion_features=None,
    missing_fill_value=2.0,
)

bin_snapping_estimator = BinSnappingEstimator(
    extractor=clone(numerai_extractions),
    top_k=20,
    n_bins=5,
    correlation_method='pandas_mean',
    abs_corr=True,
    smoothing=0.0,
    agg_mode='mean',
    era_weighting=era_weighting,
    recency_decay=recency_decay,
    prediction_transform=None,
    bias_correction=None,
    walk_forward_step_eras=None,
    walk_forward_window=walk_forward_window,
    target_horizon_eras=target_horizon_eras,
    era_stride=1,
    era_col=ERA_COL,
    target_col=TARGET_COL,
    verbose=verbose,
    random_state=random_state,
)

lag_reversal_estimator = LagReversalEstimator(
    extractor=clone(numerai_extractions),
    lag=1,
    acf_mode='feature_mean',
    acf_metric='acf',
    acf_detrend='linear',
    acf_threshold=0.0,
    sign_mode='hard',
    top_k=None,
    correlation_method='pandas_mean',
    abs_corr=True,
    center_features=True,
    era_weighting=era_weighting,
    recency_decay=recency_decay,
    prediction_transform=None,
    bias_correction=None,
    walk_forward_step_eras=None,
    walk_forward_window=walk_forward_window,
    target_horizon_eras=target_horizon_eras,
    era_stride=1,
    era_col=ERA_COL,
    target_col=TARGET_COL,
    verbose=verbose,
    random_state=random_state,
)

persistence_estimator = PersistenceEstimator(
    extractor=None,
    ticker_col=TICKER_COL,
    persistence_mode='last',
    persistence_lag=target_horizon_eras,
    smoothing=0.0,
    top_k=None,
    correlation_method=None,
    abs_corr=True,
    era_weighting=era_weighting,
    recency_decay=recency_decay,
    prediction_transform=None,
    bias_correction=None,
    walk_forward_step_eras=None,
    walk_forward_window=walk_forward_window,
    target_horizon_eras=target_horizon_eras,
    era_stride=1,
    era_col=ERA_COL,
    target_col=TARGET_COL,
    verbose=verbose,
    random_state=random_state,
)

bin_snapping_pipeline = Pipeline([('estimator', bin_snapping_estimator)])
lag_reversal_pipeline = Pipeline([('estimator', lag_reversal_estimator)])
persistence_pipeline = Pipeline([('estimator', persistence_estimator)])
print('Baseline A pipeline ready:', bin_snapping_pipeline)
print('Baseline E pipeline ready:', lag_reversal_pipeline)
print('Persistence pipeline ready:', persistence_pipeline)

Baseline A pipeline ready: Pipeline(steps=[('estimator',
                 BinSnappingEstimator(extractor=NumeraiExtractions(),
                                      target_horizon_eras=12, verbose=3))])
Baseline E pipeline ready: Pipeline(steps=[('estimator',
                 LagReversalEstimator(extractor=NumeraiExtractions(),
                                      target_horizon_eras=12, verbose=3))])
Persistence pipeline ready: Pipeline(steps=[('estimator',
                 PersistenceEstimator(persistence_lag=12,
                                      target_horizon_eras=12, verbose=3))])


# Params

## Grid

In [8]:
# SHARED `NumeraiExtractions` GRID SPACE (Used by both `BinSnappingEstimator` and `LagReversalEstimator`)
GRID_EXTRACTOR_PARAM_SPACE = {
    # ----------------------------------------------------------
    # EXTRACTOR — Era/Target Routing, Stride, Correlation & Feature Selection
    # ----------------------------------------------------------
    'estimator__extractor__era_col': [ERA_COL],
    'estimator__extractor__ticker_col': [TICKER_COL],
    'estimator__extractor__target_col': [
        TARGET_COL,
        # 'target_cyrusd_20',
        # 'target_teager2b_20',
    ],
    'estimator__extractor__year_horizon_filter': [
        None,
        # (1, 2, 3),
        # (4, 5, 6),
        # (7, 8, 9),
    ],
    'estimator__extractor__variance_threshold': [
        None,
        # 0.02,
        # 0.05,
        # 0.10,
    ],
    'estimator__extractor__correlation_method': [
        'pandas_mean',
        # None,
        # 'numerai_mean',
        # 'pearson',
        # 'spearman'
    ],
    'estimator__extractor__abs_corr': [
        True,
        # False,
    ],
    'estimator__extractor__top_k': [
        None,
        # 10,
        # 20,
        # 40,
    ],
    'estimator__extractor__feature_selection_percentile': [
        None,
        # 25.0,
        # 50.0,
        # 75.0,
        # 100.0,
    ],
    'estimator__extractor__sparse_selector': [
        None,
        # 'lasso',
        # 'elasticnet',
    ],
    'estimator__extractor__sparse_alpha': [
        0.01,
        # 0.001,
        # 0.1,
    ],
    'estimator__extractor__pca_n_components': [
        None,
        # 10,
        # 20,
        # 0.90,
    ],
    'estimator__extractor__quantize_scope': [
        'era',
        # 'train_fit',
        # 'global',
    ],
    'estimator__extractor__quantize_strategy': [
        'zscore',
        # 'quantile',
    ],
    'estimator__extractor__sequence_top_n': [
        4,
        # 2,
        # 8,
    ],
    'estimator__extractor__missing_fill_value': [
        2.0,
        # 0.0,
        # 4.0,
        # 'mean',
        # 'median',
        # 'era_mean',
        # 'era_median',
    ],

    # ----------------------------------------------------------
    # EXTRACTOR — Basic Group Stats, Fourier Harmonics & Era/Ticker Sequence Features
    # ----------------------------------------------------------
    'estimator__extractor__basic_group_stats': [
        None,
        # 'mean',
        # 'std',
        # 'skew',
        # 'mean|std',
        # 'mean|skew',
        # 'std|skew',
        # 'mean|std|skew',
    ],
    'estimator__extractor__fourier_period_eras': [
        52,
        # 26,
        # 13,
    ],
    'estimator__extractor__fourier_order': [
        None,
        # 1,
        # 2,
        # 3,
    ],
    'estimator__extractor__rolling_era_windows': [
        (4, 12),
        # (2, 6, 12),
        # (4, 8, 16),
    ],
    'estimator__extractor__include_rolling_era': [
        False,
        # True,
    ],
    'estimator__extractor__era_lag_scales': [
        (1, 2, 4),
        # (1, 2, 4, 8),
        # (2, 4, 8),
    ],
    'estimator__extractor__include_lag_diff': [
        False,
        # True,
    ],
    'estimator__extractor__ticker_features': [
        None,
        # 'rolling_mean',
        # 'momentum_diff',
        # 'weeks_since_high',
        # 'rolling_rank',
        # 'weeks_since_change',
        # 'expanding_stats',
        # 'raw_lags',
        # 'rolling_mean|momentum_diff',
        # 'rolling_mean|momentum_diff|rolling_rank',
        # 'expanding_stats|raw_lags',
        # 'rolling_mean|momentum_diff|weeks_since_high|rolling_rank|weeks_since_change|expanding_stats|raw_lags',
    ],
    'estimator__extractor__ticker_windows': [
        (4, 12),
        # (2, 6, 12),
    ],
    'estimator__extractor__ticker_lag_steps': [
        None,
        # (1, 2),
        # (1, 2, 4),
    ],
    'estimator__extractor__era_dispersion_features': [
        None,
        # 'mean',
        # 'std',
        # 'extreme_frac',
        # 'mean|std',
        # 'mean|extreme_frac',
        # 'std|extreme_frac',
        # 'mean|std|extreme_frac',
    ],
}

GRID_PARAM_SPACE_BIN_SNAPPING = {
    **GRID_EXTRACTOR_PARAM_SPACE,

    # ----------------------------------------------------------
    # MODEL — Baseline A (`BinSnappingEstimator` — Map Matching Lookup Table)
    # ----------------------------------------------------------
    'estimator__era_col': [ERA_COL],
    'estimator__target_col': [TARGET_COL],
    'estimator__correlation_method': [
        'pandas_mean',
        # None,
        # 'numerai_mean',
        # 'pearson',
        # 'spearman',
    ],
    'estimator__top_k': [
        20,
        # None,
        # 10,
        # 40,
    ],
    'estimator__n_bins': [
        5,
        # 3,
        # 7,
    ],
    'estimator__abs_corr': [
        True,
        # False,
    ],
    'estimator__smoothing': [
        0.0,
        # 25.0,
        # 50.0,
        # 100.0,
    ],
    'estimator__agg_mode': [
        'mean',
        # 'corr_weighted',
    ],
    'estimator__prediction_transform': [
        None,
        # 'era_zscore',
        # 'era_rank',
    ],
    'estimator__bias_correction': [
        None,
        # 'mean_residual',
    ],
    'estimator__walk_forward_step_eras': [
        None,
        # 10,
        # 20,
        # 40,
    ],
    'estimator__walk_forward_window': [
        walk_forward_window,
        # 'expanding',
        # 'sliding',
    ],
    'estimator__target_horizon_eras': [target_horizon_eras],
    'estimator__era_stride': [
        1,
        # 2,
        # 4,
        # 12,
    ],
    'estimator__era_weighting': [
        era_weighting,
        # None,
        # 'equal_era',
        # 'recency_linear',
        # 'recency_exp',
        # 'uniqueness',
    ],
    'estimator__recency_decay': [
        recency_decay,
        # 0.001,
        # 0.010,
    ],
    'estimator__verbose': [verbose],
    'estimator__random_state': [random_state],
}

GRID_PARAM_SPACE_LAG_REVERSAL = {
    **GRID_EXTRACTOR_PARAM_SPACE,

    # ----------------------------------------------------------
    # MODEL — Baseline E (`LagReversalEstimator` — Era Autocorrelation / PACF Sign & Shrinkage Rule)
    # ----------------------------------------------------------
    'estimator__era_col': [ERA_COL],
    'estimator__target_col': [TARGET_COL],
    'estimator__correlation_method': [
        'pandas_mean',
        # None,
        # 'numerai_mean',
        # 'pearson',
        # 'spearman',
    ],
    'estimator__lag': [
        1,
        # 2,
        # 4,
        # 6,
    ],
    'estimator__acf_mode': [
        'feature_mean',
        # 'feature_corr',
    ],
    'estimator__acf_metric': [
        'acf',
        # 'pacf',
    ],
    'estimator__acf_detrend': [
        'linear',
        # None,
        # 'constant',
    ],
    'estimator__acf_threshold': [
        0.0,
        # -0.05,
        # 0.05,
    ],
    'estimator__sign_mode': [
        'hard',
        # 'soft_threshold',
        # 'tanh',
    ],
    'estimator__top_k': [
        None,
        # 10,
        # 20,
        # 40,
    ],
    'estimator__abs_corr': [
        True,
        # False,
    ],
    'estimator__center_features': [
        True,
        # False,
    ],
    'estimator__prediction_transform': [
        None,
        # 'era_zscore',
        # 'era_rank',
    ],
    'estimator__bias_correction': [
        None,
        # 'mean_residual',
    ],
    'estimator__walk_forward_step_eras': [
        None,
        # 10,
        # 20,
        # 40,
    ],
    'estimator__walk_forward_window': [
        walk_forward_window,
        # 'expanding',
        # 'sliding',
    ],
    'estimator__target_horizon_eras': [target_horizon_eras],
    'estimator__era_stride': [
        1,
        # 2,
        # 4,
        # 12,
    ],
    'estimator__era_weighting': [
        era_weighting,
        # None,
        # 'equal_era',
        # 'recency_linear',
        # 'recency_exp',
        # 'uniqueness',
    ],
    'estimator__recency_decay': [
        recency_decay,
        # 0.001,
        # 0.010,
    ],
    'estimator__verbose': [verbose],
    'estimator__random_state': [random_state],
}

# Decoupled Persistence Grid Space (Does NOT inherit `GRID_EXTRACTOR_PARAM_SPACE`)
GRID_PARAM_SPACE_PERSISTENCE = {
    # ----------------------------------------------------------
    # MODEL — Baseline P (`PersistenceEstimator` — Brownlee Target Persistence / Zero-Rule)
    # ----------------------------------------------------------
    'estimator__era_col': [ERA_COL],
    'estimator__target_col': [TARGET_COL],
    'estimator__ticker_col': [TICKER_COL],
    'estimator__persistence_mode': [
        'last',
        # 'expanding_mean',
        # None,
    ],
    'estimator__persistence_lag': [target_horizon_eras],
    'estimator__smoothing': [
        0.0,
        # 10.0,
        # 25.0,
        # 50.0,
    ],
    'estimator__prediction_transform': [
        None,
        # 'era_zscore',
        # 'era_rank',
    ],
    'estimator__bias_correction': [
        None,
        # 'mean_residual',
    ],
    'estimator__walk_forward_step_eras': [
        None,
        # 10,
        # 20,
        # 40,
    ],
    'estimator__walk_forward_window': [
        walk_forward_window,
        # 'expanding',
        # 'sliding',
    ],
    'estimator__target_horizon_eras': [target_horizon_eras],
    'estimator__era_stride': [
        1,
        # 2,
        # 4,
        # 12,
    ],
    'estimator__era_weighting': [
        era_weighting,
        # None,
        # 'equal_era',
        # 'recency_linear',
        # 'recency_exp',
        # 'uniqueness',
    ],
    'estimator__recency_decay': [
        recency_decay,
        # 0.001,
        # 0.010,
    ],
    'estimator__verbose': [verbose],
    'estimator__random_state': [random_state],
}

## Bayesian

In [9]:
# BAYESIAN SEARCH ONLY
# Shared `BAYESIAN_EXTRACTOR_PARAM_SPACE` is used by both `BinSnappingEstimator` and `LagReversalEstimator`.
# Mode/selection parameters start with `None` (off); boolean `include_*` flags use `[False, True]`.
BAYESIAN_EXTRACTOR_PARAM_SPACE = dict()
BAYESIAN_PARAM_SPACE_BIN_SNAPPING = dict()
BAYESIAN_PARAM_SPACE_LAG_REVERSAL = dict()
BAYESIAN_PARAM_SPACE_PERSISTENCE = dict()
if SKOPT_AVAILABLE:
    BAYESIAN_EXTRACTOR_PARAM_SPACE = {
        # ----------------------------------------------------------
        # EXTRACTOR — Era/Target Routing, Stride, Correlation & Feature Selection
        # ----------------------------------------------------------
        'estimator__extractor__era_col': Categorical([ERA_COL]),
        'estimator__extractor__ticker_col': Categorical([TICKER_COL]),
        'estimator__extractor__target_col': Categorical([
            TARGET_COL,
        ]),
        'estimator__extractor__year_horizon_filter': Categorical([
            None,
            (1, 2, 3),
            (4, 5, 6),
            (7, 8, 9),
        ]),
        'estimator__extractor__variance_threshold': Categorical([
            None,
            0.02,
            0.05,
            0.10,
        ]),
        'estimator__extractor__correlation_method': Categorical([
            None,
            'pandas_mean',
            'numerai_mean',
            'pearson',
            'spearman',
        ]),
        'estimator__extractor__abs_corr': Categorical([True, False]),
        'estimator__extractor__top_k': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__extractor__feature_selection_percentile': Categorical([
            None,
            25.0,
            50.0,
            75.0,
            100.0,
        ]),
        'estimator__extractor__sparse_selector': Categorical([
            None,
            'lasso',
            'elasticnet',
        ]),
        'estimator__extractor__sparse_alpha': Real(1e-4, 1e-1, prior='log-uniform'),
        'estimator__extractor__pca_n_components': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__extractor__quantize_scope': Categorical([
            'era',
            'train_fit',
            'global',
        ]),
        'estimator__extractor__quantize_strategy': Categorical([
            'zscore',
            'quantile',
        ]),
        'estimator__extractor__sequence_top_n': Categorical([
            2,
            4,
            8,
        ]),
        'estimator__extractor__missing_fill_value': Categorical([
            2.0,
            0.0,
            4.0,
            'mean',
            'median',
            'era_mean',
            'era_median',
        ]),

        # ----------------------------------------------------------
        # EXTRACTOR — Basic Group Stats, Fourier Harmonics & Era/Ticker Sequence Features
        # ----------------------------------------------------------
        'estimator__extractor__basic_group_stats': Categorical([
            None,
            'mean',
            'std',
            'skew',
            'mean|std',
            'mean|skew',
            'std|skew',
            'mean|std|skew',
        ]),
        'estimator__extractor__fourier_period_eras': Categorical([
            52,
            26,
            13,
        ]),
        'estimator__extractor__fourier_order': Categorical([
            None,
            1,
            2,
            3,
        ]),
        'estimator__extractor__rolling_era_windows': Categorical([
            (4, 12),
            (2, 6, 12),
            (4, 8, 16),
        ]),
        'estimator__extractor__include_rolling_era': Categorical([False, True]),
        'estimator__extractor__era_lag_scales': Categorical([
            (1, 2, 4),
            (1, 2, 4, 8),
            (2, 4, 8),
        ]),
        'estimator__extractor__include_lag_diff': Categorical([False, True]),
        'estimator__extractor__ticker_features': Categorical([
            None,
            'rolling_mean',
            'momentum_diff',
            'weeks_since_high',
            'rolling_rank',
            'weeks_since_change',
            'expanding_stats',
            'raw_lags',
            'rolling_mean|momentum_diff',
            'rolling_mean|momentum_diff|rolling_rank',
            'expanding_stats|raw_lags',
            'rolling_mean|momentum_diff|weeks_since_high|rolling_rank|weeks_since_change|expanding_stats|raw_lags',
        ]),
        'estimator__extractor__ticker_windows': Categorical([
            (4, 12),
            (2, 6, 12),
        ]),
        'estimator__extractor__ticker_lag_steps': Categorical([
            None,
            (1, 2),
            (1, 2, 4),
        ]),
        'estimator__extractor__era_dispersion_features': Categorical([
            None,
            'mean',
            'std',
            'extreme_frac',
            'mean|std',
            'mean|extreme_frac',
            'std|extreme_frac',
            'mean|std|extreme_frac',
        ]),
    }

    BAYESIAN_PARAM_SPACE_BIN_SNAPPING = {
        **BAYESIAN_EXTRACTOR_PARAM_SPACE,

        # ----------------------------------------------------------
        # MODEL — Baseline A (`BinSnappingEstimator` — Map Matching Lookup Table)
        # ----------------------------------------------------------
        'estimator__era_col': Categorical([ERA_COL]),
        'estimator__target_col': Categorical([TARGET_COL]),
        'estimator__correlation_method': Categorical([
            None,
            'pandas_mean',
            'numerai_mean',
            'pearson',
            'spearman',
        ]),
        'estimator__top_k': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__n_bins': Categorical([
            5,
            3,
            7,
        ]),
        'estimator__abs_corr': Categorical([True, False]),
        'estimator__smoothing': Real(0.0, 100.0),
        'estimator__agg_mode': Categorical(['mean', 'corr_weighted']),
        'estimator__prediction_transform': Categorical([
            None,
            'era_zscore',
            'era_rank',
        ]),
        'estimator__bias_correction': Categorical([
            None,
            'mean_residual',
        ]),
        'estimator__walk_forward_step_eras': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__walk_forward_window': Categorical([
            'expanding',
            'sliding',
        ]),
        'estimator__target_horizon_eras': Categorical([target_horizon_eras]),
        'estimator__era_stride': Categorical([1]),
        'estimator__era_weighting': Categorical([
            None,
            'equal_era',
            'recency_linear',
            'recency_exp',
            'uniqueness',
            'uniqueness_decay',
        ]),
        'estimator__recency_decay': Real(0.001, 0.020),
        'estimator__verbose': Categorical([verbose]),
        'estimator__random_state': Categorical([random_state]),
    }
    BAYESIAN_PARAM_SPACE_BIN_SNAPPING = build_conditional_bayesian_spaces(
        drop_constant_dims(BAYESIAN_PARAM_SPACE_BIN_SNAPPING, bin_snapping_pipeline),
        total_n_iter=n_iter,
    )

    BAYESIAN_PARAM_SPACE_LAG_REVERSAL = {
        **BAYESIAN_EXTRACTOR_PARAM_SPACE,

        # ----------------------------------------------------------
        # MODEL — Baseline E (`LagReversalEstimator` — Era Autocorrelation / PACF Sign & Shrinkage Rule)
        # ----------------------------------------------------------
        'estimator__era_col': Categorical([ERA_COL]),
        'estimator__target_col': Categorical([TARGET_COL]),
        'estimator__correlation_method': Categorical([
            None,
            'pandas_mean',
            'numerai_mean',
            'pearson',
            'spearman',
        ]),
        'estimator__lag': Integer(1, 6),
        'estimator__acf_mode': Categorical(['feature_mean', 'feature_corr']),
        'estimator__acf_metric': Categorical(['acf', 'pacf']),
        'estimator__acf_detrend': Categorical([
            None,
            'linear',
            'constant',
        ]),
        'estimator__acf_threshold': Real(-0.20, 0.20),
        'estimator__sign_mode': Categorical([
            'hard',
            'soft_threshold',
            'tanh',
        ]),
        'estimator__top_k': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__abs_corr': Categorical([True, False]),
        'estimator__center_features': Categorical([True, False]),
        'estimator__prediction_transform': Categorical([
            None,
            'era_zscore',
            'era_rank',
        ]),
        'estimator__bias_correction': Categorical([
            None,
            'mean_residual',
        ]),
        'estimator__walk_forward_step_eras': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__walk_forward_window': Categorical([
            'expanding',
            'sliding',
        ]),
        'estimator__target_horizon_eras': Categorical([target_horizon_eras]),
        'estimator__era_stride': Categorical([1]),
        'estimator__era_weighting': Categorical([
            None,
            'equal_era',
            'recency_linear',
            'recency_exp',
            'uniqueness',
            'uniqueness_decay',
        ]),
        'estimator__recency_decay': Real(0.001, 0.020),
        'estimator__verbose': Categorical([verbose]),
        'estimator__random_state': Categorical([random_state]),
    }
    BAYESIAN_PARAM_SPACE_LAG_REVERSAL = build_conditional_bayesian_spaces(
        drop_constant_dims(BAYESIAN_PARAM_SPACE_LAG_REVERSAL, lag_reversal_pipeline),
        total_n_iter=n_iter,
    )

    # Decoupled Persistence Bayesian Space (Does NOT inherit `BAYESIAN_EXTRACTOR_PARAM_SPACE`)
    BAYESIAN_PARAM_SPACE_PERSISTENCE = {
        # ----------------------------------------------------------
        # MODEL — Baseline P (`PersistenceEstimator` — Brownlee Target Persistence / Zero-Rule)
        # ----------------------------------------------------------
        'estimator__era_col': Categorical([ERA_COL]),
        'estimator__target_col': Categorical([TARGET_COL]),
        'estimator__ticker_col': Categorical([TICKER_COL]),
        'estimator__persistence_mode': Categorical([
            None,
            'last',
            'expanding_mean',
        ]),
        'estimator__persistence_lag': Categorical([target_horizon_eras]),
        'estimator__smoothing': Real(0.0, 100.0),
        'estimator__prediction_transform': Categorical([
            None,
            'era_zscore',
            'era_rank',
        ]),
        'estimator__bias_correction': Categorical([
            None,
            'mean_residual',
        ]),
        'estimator__walk_forward_step_eras': Categorical([
            None,
            20,
            40,
        ]),
        'estimator__walk_forward_window': Categorical([
            'expanding',
            'sliding',
        ]),
        'estimator__target_horizon_eras': Categorical([target_horizon_eras]),
        'estimator__era_stride': Categorical([1]),
        'estimator__era_weighting': Categorical([
            None,
            'equal_era',
            'recency_linear',
            'recency_exp',
            'uniqueness',
            'uniqueness_decay',
        ]),
        'estimator__recency_decay': Real(0.001, 0.020),
        'estimator__verbose': Categorical([verbose]),
        'estimator__random_state': Categorical([random_state]),
    }
    BAYESIAN_PARAM_SPACE_PERSISTENCE = build_conditional_bayesian_spaces(
        drop_constant_dims(BAYESIAN_PARAM_SPACE_PERSISTENCE, persistence_pipeline),
        total_n_iter=n_iter,
    )

bin_snapping_param_names = set(bin_snapping_pipeline.get_params(deep=True))
lag_reversal_param_names = set(lag_reversal_pipeline.get_params(deep=True))
persistence_param_names = set(persistence_pipeline.get_params(deep=True))
assert set(GRID_PARAM_SPACE_BIN_SNAPPING).issubset(bin_snapping_param_names)
assert set(GRID_PARAM_SPACE_LAG_REVERSAL).issubset(lag_reversal_param_names)
assert set(GRID_PARAM_SPACE_PERSISTENCE).issubset(persistence_param_names)
for _bspace, _pnames in (
    (BAYESIAN_PARAM_SPACE_BIN_SNAPPING, bin_snapping_param_names),
    (BAYESIAN_PARAM_SPACE_LAG_REVERSAL, lag_reversal_param_names),
    (BAYESIAN_PARAM_SPACE_PERSISTENCE, persistence_param_names),
):
    _bkeys = set().union(*(sub.keys() for sub, _ in _bspace)) if isinstance(_bspace, list) else set(_bspace)
    assert _bkeys.issubset(_pnames)

if search_mode == 'bayes':
    bin_snapping_param_space = BAYESIAN_PARAM_SPACE_BIN_SNAPPING
    lag_reversal_param_space = BAYESIAN_PARAM_SPACE_LAG_REVERSAL
    persistence_param_space = BAYESIAN_PARAM_SPACE_PERSISTENCE
else:
    bin_snapping_param_space = GRID_PARAM_SPACE_BIN_SNAPPING
    lag_reversal_param_space = GRID_PARAM_SPACE_LAG_REVERSAL
    persistence_param_space = GRID_PARAM_SPACE_PERSISTENCE
print(f'Active search_mode={search_mode} | baseline_mode={baseline_mode}')

Active search_mode=bayes | baseline_mode=all


# Train

In [10]:
_train_t0 = time.perf_counter()
baseline_results = {}
search_summary_rows = []
cv_results_by_baseline = {}
fitted_default_models = {}

error_bin_cfg = {
    'mode': error_bin_mode,
    'pct_bins': error_pct_bins,
    'zero_tols': error_zero_tols,
    'abs_bins': error_abs_bins,
}

persistence_jobs = []
if baseline_mode in ('persistence', 'all'):
    persistence_jobs.append(('Baseline P (Persistence)', persistence_pipeline, persistence_param_space))

heavy_jobs = []
if baseline_mode in ('baseline_a', 'both', 'all'):
    heavy_jobs.append(('Baseline A (Bin-Snapping)', bin_snapping_pipeline, bin_snapping_param_space))
if baseline_mode in ('baseline_e', 'both', 'all'):
    heavy_jobs.append(('Baseline E (Lag-Reversal)', lag_reversal_pipeline, lag_reversal_param_space))

_cpu_count = os.cpu_count() or 1
_job_budget = _cpu_count if int(n_jobs) == -1 else max(1, int(n_jobs))
outer_jobs = min(_job_budget, len(heavy_jobs)) if heavy_jobs else 1

effective_cache_max_entries = (
    min(int(extraction_cache_max_entries), 2)
    if (outer_jobs > 1 and str(feature_set).strip().lower() in ('medium', 'all'))
    else int(extraction_cache_max_entries)
)
extraction_cache_cfg = {
    'enabled': extraction_cache_enabled,
    'max_entries': effective_cache_max_entries,
}

common_run_kwargs = dict(
    search_mode=search_mode,
    score_version=score_version,
    eval_mode=eval_mode,
    cv_object=cv_object,
    n_iter=n_iter,
    random_state=random_state,
    verbose=verbose,
    error_score=error_score,
    n_jobs=1,
    walk_forward_step_eras=walk_forward_step_eras,
    walk_forward_window=walk_forward_window,
    target_horizon_eras=target_horizon_eras,
    era_stride=era_stride,
    error_bin_config=error_bin_cfg,
    extraction_cache_config=extraction_cache_cfg,
)

# 1. Run Baseline P (Persistence) first in the main process on metadata-only columns (0 feature copies)
persistence_payloads = []
if persistence_jobs:
    persist_cols = [c for c in (ERA_COL, TICKER_COL) if c in X_train.columns]
    X_train_persist = X_train[persist_cols]
    X_val_persist = X_validation[persist_cols]
    print(f'Running Baseline P (Persistence) in main process on {persist_cols} (0 feature columns copied)')
    for b_name, b_pipe, b_space in persistence_jobs:
        persistence_payloads.append(
            run_baseline(
                b_name, b_pipe, b_space,
                X_train_persist, y_train, train_eras,
                X_val_persist, y_validation, validation_eras,
                [],
                **common_run_kwargs,
            )
        )
    del X_train_persist, X_val_persist

# 2. Run heavy feature baselines (A & E) in parallel via loky with memmap (max_nbytes='10M', at most 2 workers)
heavy_payloads = []
if heavy_jobs:
    print(
        f'Running {len(heavy_jobs)} heavy baseline(s) with outer_jobs={outer_jobs} '
        f'(n_jobs={n_jobs}, cpu_count={_cpu_count}, inner cv_n_jobs=1, extraction_cache_max_entries={effective_cache_max_entries})'
    )
    if outer_jobs > 1:
        heavy_payloads = Parallel(n_jobs=outer_jobs, backend='loky', max_nbytes='10M')(
            delayed(run_baseline)(
                b_name, b_pipe, b_space,
                X_train, y_train, train_eras,
                X_validation, y_validation, validation_eras,
                feature_columns,
                **common_run_kwargs,
            )
            for b_name, b_pipe, b_space in heavy_jobs
        )
    else:
        heavy_payloads = [
            run_baseline(
                b_name, b_pipe, b_space,
                X_train, y_train, train_eras,
                X_validation, y_validation, validation_eras,
                feature_columns,
                **common_run_kwargs,
            )
            for b_name, b_pipe, b_space in heavy_jobs
        ]

# Merge in original A -> E -> P order
payloads = list(heavy_payloads) + list(persistence_payloads)
for b_name, b_results, b_cv_df, b_summary_row, b_default_model in payloads:
    baseline_results.update(b_results)
    cv_results_by_baseline[b_name] = b_cv_df
    search_summary_rows.append(b_summary_row)
    fitted_default_models[b_name] = b_default_model

train_elapsed_s = time.perf_counter() - _train_t0
print(f'Completed {len(payloads)} baseline(s) in {train_elapsed_s:.1f}s (heavy outer_jobs={outer_jobs})')

Running Baseline P (Persistence) in main process on ['era', 'numerai_ticker'] (0 feature columns copied)


--- Baseline P (Persistence) — Final Training Scores ---
Percentage Summary | Numerai CORR: +2.07% | Positive Eras: 52.1% | Directional Accuracy: 41.5%
Numerai CORR: 0.0207
Spearman Correlation: 0.0250
Average Payout: 0.0978
Sharpe Ratio (Numerai / Spearman): 0.1244 / 0.1579 | PSR (SR > 0): 0.9639
Cumulative CORR Drawdown (CORR pts): 1.5037
Mean Absolute Error (MAE): 0.1825
Forecast Bias (mean(y - y_hat)): -0.0019
Directional Hit Rate (vs 0.5): 0.4154 (41.5%)
Era CORR White-Noise & Regime Tests | Lag-1 ACF: 0.0369 | Ljung-Box p: 0.5500 | ARCH-LM p: 0.4470 | ADF p: 0.0000
Era CORR Tail Risk | Skew: 0.0923 | Excess Kurtosis: 0.1975 | Jarque-Bera p: 0.7681
[SANITY CHECK PASSED] numerai_corr=0.0207 is within healthy [0.005, 0.040] Numerai range.


--- Baseline P (Persistence) — Final Validation Scores ---
Percentage Summary | Numerai CORR: -0.08% | Positive Eras: 45.4% | Directional Accuracy: 39.3%
Numerai CORR: -0.0008
Spearman Correlation: 0.0019
Average Payout: 0.0095
Sharpe Ratio (Numerai / Spearman): -0.0640 / 0.1501 | PSR (SR > 0): 0.1009
Cumulative CORR Drawdown (CORR pts): 0.8642
Mean Absolute Error (MAE): 0.1905
Forecast Bias (mean(y - y_hat)): -0.0015
Directional Hit Rate (vs 0.5): 0.3932 (39.3%)
Era CORR White-Noise & Regime Tests | Lag-1 ACF: 0.7664 | Ljung-Box p: 0.0000 | ARCH-LM p: 0.0000 | ADF p: 0.0000
Era CORR Tail Risk | Skew: -0.0371 | Excess Kurtosis: 0.4476 | Jarque-Bera p: 0.2099
[SANITY WARNING — WEAK/BROKEN SIGNAL] numerai_corr=-0.0008 < 0.005 floor (check alignment, target, or over-regularization).


--- Baseline P (Persistence) — Walk-Forward Validation Scores ---
Percentage Summary | Numerai CORR: +0.37% | Positive Eras: 56.9% | Directional Accuracy: 30.5%
Numerai CORR: 0.0037
Spearman Correlation: 0.0024
Average Payout: 0.0118
Sharpe Ratio (Numerai / Spearman): 0.1624 / 0.1104 | PSR (SR > 0): 0.9994
Cumulative CORR Drawdown (CORR pts): 0.8651
Mean Absolute Error (MAE): 0.2177
Forecast Bias (mean(y - y_hat)): -0.0010
Directional Hit Rate (vs 0.5): 0.3050 (30.5%)
Era CORR White-Noise & Regime Tests | Lag-1 ACF: 0.8586 | Ljung-Box p: 0.0000 | ARCH-LM p: 0.0000 | ADF p: 0.0000
Era CORR Tail Risk | Skew: 0.0820 | Excess Kurtosis: -0.3447 | Jarque-Bera p: 0.2803
[SANITY WARNING — WEAK/BROKEN SIGNAL] numerai_corr=0.0037 < 0.005 floor (check alignment, target, or over-regularization).


Fitting 1 folds for each of 1 candidates, totalling 1 fits
  [Fold Score | v1_skepta] Score: 0.0000 | Numerai CORR: +0.00% (0.0000, std=0.0000) | Sharpe: 0.00 | CORR DD: 0.0000 | Positive Eras: 0.0% | Dir Acc: 50.6% | MAE: 0.1565
[CV 1/1] END estimator__bias_correction=mean_residual, estimator__era_stride=12, estimator__era_weighting=equal_era, estimator__persistence_mode=None, estimator__prediction_transform=era_rank, estimator__recency_decay=0.0021839458760373577, estimator__smoothing=0.0, estimator__walk_forward_step_eras=20, estimator__walk_forward_window=sliding;, score=0.000 total time=   0.0s


Fitting 1 folds for each of 1 candidates, totalling 1 fits


... (38 outputs left out)

  [Fold Score | v1_skepta] Score: -0.0371 | Numerai CORR: +0.79% (0.0079, std=0.0901) | Sharpe: 0.09 | CORR DD: 0.4616 | Positive Eras: 50.0% | Dir Acc: 51.2% | MAE: 0.5435
[CV 1/1] END estimator__abs_corr=False, estimator__acf_detrend=constant, estimator__acf_metric=acf, estimator__acf_mode=feature_mean, estimator__acf_threshold=-0.05873763389572492, estimator__bias_correction=None, estimator__center_features=False, estimator__correlation_method=pearson, estimator__era_stride=5, estimator__era_weighting=recency_exp, estimator__extractor__abs_corr=False, estimator__extractor__basic_group_stats=std, estimator__extractor__correlation_method=numerai_mean, estimator__extractor__era_dispersion_features=mean|std|extreme_frac, estimator__extractor__era_lag_scales=[1, 2, 4, 8], estimator__extractor__feature_selection_percentile=75.0, estimator__extractor__fourier_order=3, estimator__extractor__fourier_period_eras=26, estimator__extractor__include_lag_diff=True, estimator__extractor__include_ro

--- Baseline E (Lag-Reversal) [Tuned] — Final Training Scores ---
Percentage Summary | Numerai CORR: -1.03% | Positive Eras: 47.9% | Directional Accuracy: 49.3%
Numerai CORR: -0.0103
Spearman Correlation: -0.0038
Average Payout: 0.0002
Sharpe Ratio (Numerai / Spearman): -0.0734 / -0.0262 | PSR (SR > 0): 0.1463
Cumulative CORR Drawdown (CORR pts): 3.7372
Mean Absolute Error (MAE): 0.5688
Forecast Bias (mean(y - y_hat)): 0.5666
Directional Hit Rate (vs 0.5): 0.4928 (49.3%)
Era CORR White-Noise & Regime Tests | Lag-1 ACF: -0.0336 | Ljung-Box p: 0.6867 | ARCH-LM p: 0.7077 | ADF p: 0.0000
Era CORR Tail Risk | Skew: 0.1148 | Excess Kurtosis: -0.1203 | Jarque-Bera p: 0.7268
Feature Exposure: 0.0644
[SANITY WARNING — WEAK/BROKEN SIGNAL] numerai_corr=-0.0103 < 0.005 floor (check alignment, target, or over-regularization).


--- Baseline E (Lag-Reversal) [Tuned] — Final Validation Scores ---
Percentage Summary | Numerai CORR: -0.18% | Positive Eras: 49.9% | Directional Accuracy: 50.0%
Numerai CORR: -0.0018
Spearman Correlation: -0.0001
Average Payout: -0.0005
Sharpe Ratio (Numerai / Spearman): -0.1379 / -0.0071 | PSR (SR > 0): 0.0023
Cumulative CORR Drawdown (CORR pts): 1.0281
Mean Absolute Error (MAE): 0.5735
Forecast Bias (mean(y - y_hat)): 0.5714
Directional Hit Rate (vs 0.5): 0.5003 (50.0%)
Era CORR White-Noise & Regime Tests | Lag-1 ACF: 0.8153 | Ljung-Box p: 0.0000 | ARCH-LM p: 0.0000 | ADF p: 0.0001
Era CORR Tail Risk | Skew: -0.4690 | Excess Kurtosis: -0.0894 | Jarque-Bera p: 0.0006
Feature Exposure: 0.0696
[SANITY WARNING — WEAK/BROKEN SIGNAL] numerai_corr=-0.0018 < 0.005 floor (check alignment, target, or over-regularization).


--- Baseline E (Lag-Reversal) [Tuned] — Walk-Forward Validation Scores ---
Percentage Summary | Numerai CORR: -0.38% | Positive Eras: 37.6% | Directional Accuracy: 50.0%
Numerai CORR: -0.0038
Spearman Correlation: -0.0020
Average Payout: -0.0102
Sharpe Ratio (Numerai / Spearman): -0.2896 / -0.1531 | PSR (SR > 0): 0.0000
Cumulative CORR Drawdown (CORR pts): 1.7295
Mean Absolute Error (MAE): 0.4844
Forecast Bias (mean(y - y_hat)): 0.4804
Directional Hit Rate (vs 0.5): 0.5000 (50.0%)
Era CORR White-Noise & Regime Tests | Lag-1 ACF: 0.7087 | Ljung-Box p: 0.0000 | ARCH-LM p: 0.0000 | ADF p: 0.0000
Era CORR Tail Risk | Skew: 0.0777 | Excess Kurtosis: -0.2340 | Jarque-Bera p: 0.4954
Feature Exposure: 0.0148
[SANITY WARNING — WEAK/BROKEN SIGNAL] numerai_corr=-0.0038 < 0.005 floor (check alignment, target, or over-regularization).
Completed 3 baseline(s) in 856.9s (heavy outer_jobs=1)


In [11]:
comparison_columns = [
    'variant',
    'score_version',
    'train_corr',
    'validation_corr',
    'validation_spearman',
    'validation_payout',
    'era_consistency_pct',
    'validation_sharpe',
    'validation_lo_sharpe',
    'validation_psr',
    'validation_dsr',
    'effective_n_trials_onc',
    'validation_fnc',
    'validation_fnc_sharpe',
    'tuw_max_eras',
    'hhi_pos',
    'hhi_neg',
    'validation_spearman_sharpe',
    'validation_mae',
    'forecast_bias',
    'directional_hit_rate',
    'era_corr_lag1_acf',
    'ljung_box_pvalue',
    'arch_lm_pvalue',
    'adf_pvalue',
    'era_corr_skew',
    'era_corr_kurtosis',
    'jarque_bera_pvalue',
    'feature_exposure',
    'corr_drawdown',
    'start_capital',
    'end_capital',
    'total_profit',
    'total_return_pct',
    'money_drawdown_pct',
    'max_drawdown_gbp',
    'worst_round_gbp',
    'corr_gap',
    'sanity_status',
]
comparison_df = (
    pd.DataFrame([{column: result.get(column, np.nan) for column in comparison_columns} for result in baseline_results.values()])
    .sort_values(['validation_sharpe', 'validation_corr'], ascending=False, kind='stable')
    .reset_index(drop=True)
)
search_summary_df = (
    pd.DataFrame(search_summary_rows)
    .sort_values('best_cv_score', ascending=False, kind='stable')
    .reset_index(drop=True)
)

# Select the locked winner strictly from Train CV (never pick on held-out validation!)
cv_winner_baseline = str(search_summary_df.loc[0, 'baseline'])
preferred_locked_variants = [
    f'{cv_winner_baseline} [Tuned]',
    f'{cv_winner_baseline} [Tuned Walk-Forward]',
    f'{cv_winner_baseline} [Default]',
]
best_variant_name = next(
    (v for v in preferred_locked_variants if v in baseline_results),
    str(comparison_df.loc[0, 'variant']),
)
best_validation_era_corrs = baseline_results[best_variant_name]['validation_era_corrs']
best_validation_corr = baseline_results[best_variant_name]['validation_corr']

print('Held-Out Validation Comparison (One-Wallet Staking + Split CORR vs Money Drawdowns):')
display(comparison_df)
print(f'Train CV Search Summary ({search_mode}, score_version={score_version}, refit=False):')
display(search_summary_df)
print(f'Train-CV-Locked Winner (evaluated once on held-out validation): {best_variant_name}')

Held-Out Validation Comparison (One-Wallet Staking + Split CORR vs Money Drawdowns):


,variant,score_version,train_corr,validation_corr,validation_spearman,validation_payout,validation_sharpe,validation_psr,validation_dsr,validation_spearman_sharpe,...,corr_drawdown,start_capital,end_capital,total_profit,total_return_pct,money_drawdown_pct,max_drawdown_gbp,worst_round_gbp,corr_gap,sanity_status
0,Baseline A (Bin-Snapping) [Walk-Forward],v1_skepta,0.0859,0.0039,0.0057,0.0283,0.3206,1.0000,1.0000,0.4154,...,0.2851,1000.0,1075.48,75.48,7.55,-1.33,-14.40,-1.37,0.0820,LOW_SIGNAL_WARNING
1,Baseline P (Persistence) [Walk-Forward],v1_skepta,0.0207,0.0037,0.0024,0.0118,0.1624,0.9994,0.9994,0.1104,...,0.8651,1000.0,1070.70,70.70,7.07,-4.02,-43.16,-2.75,0.0170,LOW_SIGNAL_WARNING
2,Baseline A (Bin-Snapping) [Tuned Walk-Forward],v1_skepta,0.0061,0.0016,0.0025,0.0127,0.1256,0.9944,0.4902,0.1927,...,0.4991,1000.0,1029.34,29.34,2.93,-2.32,-24.21,-1.50,0.0045,LOW_SIGNAL_WARNING
3,Baseline A (Bin-Snapping) [Tuned],v1_skepta,0.0061,0.0005,-0.0000,-0.0002,0.0358,0.7630,0.0342,-0.0034,...,0.7381,1000.0,1008.49,8.49,0.85,-3.38,-35.28,-1.65,0.0056,LOW_SIGNAL_WARNING
4,Baseline A (Bin-Snapping) [Default],v1_skepta,0.0859,0.0000,0.0004,0.0018,0.0016,0.5126,0.5126,0.0276,...,0.8326,1000.0,999.72,-0.28,-0.03,-3.85,-38.94,-1.49,0.0859,LOW_SIGNAL_WARNING
5,Baseline P (Persistence) [Tuned],v1_skepta,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,1000.0,1000.00,0.00,0.00,0.00,0.00,0.00,0.0000,LOW_SIGNAL_WARNING
6,Baseline P (Persistence) [Tuned Walk-Forward],v1_skepta,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,1000.0,1000.00,0.00,0.00,0.00,0.00,0.00,0.0000,LOW_SIGNAL_WARNING
7,Baseline P (Persistence) [Default],v1_skepta,0.0207,-0.0008,0.0019,0.0095,-0.0640,0.1009,0.1009,0.1501,...,0.8642,1000.0,984.56,-15.44,-1.54,-3.99,-40.88,-1.87,0.0215,LOW_SIGNAL_WARNING
8,Baseline E (Lag-Reversal) [Tuned],v1_skepta,-0.0103,-0.0018,-0.0001,-0.0005,-0.1379,0.0023,0.0000,-0.0071,...,1.0281,1000.0,965.26,-34.74,-3.47,-4.67,-47.15,-2.10,-0.0085,LOW_SIGNAL_WARNING
9,Baseline E (Lag-Reversal) [Default],v1_skepta,-0.0274,-0.0027,-0.0030,-0.0149,-0.1935,0.0000,0.0000,-0.1967,...,1.6536,1000.0,950.54,-49.46,-4.95,-7.52,-75.64,-1.91,-0.0247,LOW_SIGNAL_WARNING


Train CV Search Summary (bayes, score_version=v1_skepta, refit=False):


,baseline,search_mode,score_version,best_cv_score,best_cv_corr,best_cv_score_pct,best_params
0,Baseline P (Persistence),bayes,v1_skepta,0.000000,0.0000,0.00,{'estimator__bias_correction': 'mean_residual'...
1,Baseline A (Bin-Snapping),bayes,v1_skepta,-0.027849,0.0355,-2.78,"{'estimator__abs_corr': True, 'estimator__agg_..."
2,Baseline E (Lag-Reversal),bayes,v1_skepta,-0.037143,0.0079,-3.71,"{'estimator__abs_corr': False, 'estimator__acf..."


Train-CV-Locked Winner (evaluated once on held-out validation): Baseline P (Persistence) [Tuned]


# Analysis

In [12]:
era_positions = np.arange(len(best_validation_era_corrs))
tick_step = max(1, len(best_validation_era_corrs) // 10)
tick_labels = best_validation_era_corrs.index.astype(str)[::tick_step]
line_colors = ['#2563eb', '#10b981', '#f59e0b', '#8b5cf6']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.bar(
    era_positions,
    best_validation_era_corrs.to_numpy(),
    color=['#10b981' if era_corr >= 0 else '#ef4444' for era_corr in best_validation_era_corrs.to_numpy()],
    alpha=0.85,
)
ax.axhline(0.0, color='#64748b', linewidth=1.0)
ax.axhline(best_validation_corr, color='#1d4ed8', linestyle='--', linewidth=1.5, label=f'Winner Mean = {best_validation_corr:.4f}')
ax.set_xticks(era_positions[::tick_step])
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Validation Era')
ax.set_ylabel('Numerai CORR')
ax.set_title(f'Per-Era Validation CORR ({best_variant_name})')
ax.legend()

ax = axes[1]
for color_index, (variant_name, result) in enumerate(baseline_results.items()):
    line_color = line_colors[color_index % len(line_colors)]
    ax.plot(
        era_positions,
        result['validation_era_corrs'].cumsum().to_numpy(),
        color=line_color,
        linewidth=2.0,
        label=f"{variant_name} (Sharpe {result['validation_sharpe']:.2f})",
    )
ax.axhline(0.0, color='#64748b', linewidth=0.8)
ax.set_xticks(era_positions[::tick_step])
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Validation Era')
ax.set_ylabel('Cumulative CORR')
ax.set_title('Cumulative Validation CORR Across Default Baselines')
ax.legend(fontsize=8)

plt.tight_layout()
if save_results:
    results_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(results_dir / f'{timestamp}_{experiment_name}_era_perf.png', dpi=140)
    comparison_df.to_csv(results_dir / f'{timestamp}_{experiment_name}_comparison.csv', index=False)
if export_diagnostics or save_results:
    results_dir.mkdir(parents=True, exist_ok=True)
    diagnostics_parquet_path = export_diagnostics_predictions(
        val_diag_meta,
        baseline_results[best_variant_name]['validation_preds'],
        results_dir / 'validation_predictions.parquet',
        era_col=ERA_COL,
    )
    print('Exported Numerai Diagnostics predictions Parquet (MMC & FNC):', diagnostics_parquet_path)
plt.show()

# Brownlee Residual, White-Noise & Tail-Normality Diagnostics (Per-Era Mean Residual + Era CORR Autocorrelation + Residual Q-Q Plot)
winner_val_preds = np.asarray(baseline_results[best_variant_name]['validation_preds'], dtype=np.float64).ravel()
winner_residuals = pd.Series(y_validation.to_numpy(dtype=np.float64) - winner_val_preds, index=validation_eras.index)
per_era_mean_residuals = winner_residuals.groupby(validation_eras, sort=True).mean()

fig_res, axes_res = plt.subplots(1, 3, figsize=(20, 4.5))
axes_res[0].plot(era_positions, per_era_mean_residuals.to_numpy(), color='#dc2626', marker='o', markersize=3.5, linewidth=1.6, label='Mean Residual (y - y_hat)')
axes_res[0].axhline(0.0, color='#64748b', linestyle='--', linewidth=1.2, label='Zero Bias (0.0)')
axes_res[0].set_xticks(era_positions[::tick_step])
axes_res[0].set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
axes_res[0].set_xlabel('Validation Era')
axes_res[0].set_ylabel('Mean Forecast Residual')
axes_res[0].set_title(f'Per-Era Mean Forecast Residual ({best_variant_name})')
axes_res[0].legend()

if len(best_validation_era_corrs) >= 2:
    autocorrelation_plot(best_validation_era_corrs, ax=axes_res[1], color='#2563eb')
axes_res[1].set_title(f'White-Noise Check: Per-Era CORR Autocorrelation ({best_variant_name})')

if len(per_era_mean_residuals) >= 3:
    qqplot(per_era_mean_residuals.to_numpy(dtype=np.float64), line='45', fit=True, ax=axes_res[2])
axes_res[2].set_title(f'Residual Q-Q Plot ({best_variant_name})')
plt.tight_layout()
if save_results:
    plt.savefig(results_dir / f'{timestamp}_{experiment_name}_residual_white_noise.png', dpi=140)
plt.show()

Exported Numerai Diagnostics predictions Parquet (MMC & FNC): C:\Users\maran\OneDrive\Documents\Git Profile\Data-Projects\numerai\results_numerai_v6.0_all_eras_stride4_baselines_ae_target_bayes\validation_predictions.parquet


In [13]:
if 'Baseline A (Bin-Snapping)' in fitted_default_models:
    fitted_bin_snapping = fitted_default_models['Baseline A (Bin-Snapping)'].named_steps['estimator']
    lookup_table_df = pd.DataFrame(fitted_bin_snapping.lookup_table_).T
    lookup_table_df.columns = [f'bin_{bin_index}' for bin_index in lookup_table_df.columns]
    lookup_table_df['train_corr'] = fitted_bin_snapping.selected_corrs_
    print(f'Baseline A Learned Lookup Table (top {len(lookup_table_df)} features):')
    display(lookup_table_df.head(15))

if 'Baseline E (Lag-Reversal)' in fitted_default_models:
    fitted_lag_reversal = fitted_default_models['Baseline E (Lag-Reversal)'].named_steps['estimator']
    n_kept_signals = int((fitted_lag_reversal.inversion_signs_ > 0).sum())
    n_inverted_signals = int((fitted_lag_reversal.inversion_signs_ < 0).sum())
    print(f'Baseline E Signal Rules | kept (+1)={n_kept_signals} | inverted (-1)={n_inverted_signals}')
    display(fitted_lag_reversal.rule_summary_.head(20))

if 'Baseline P (Persistence)' in fitted_default_models:
    fitted_persistence = fitted_default_models['Baseline P (Persistence)'].named_steps['estimator']
    print(
        f'Baseline P (Persistence) | mode={fitted_persistence.persistence_mode} | '
        f'lag={fitted_persistence.persistence_lag} | global_mean={fitted_persistence.global_mean_:.4f} | '
        f'tracked_tickers={len(fitted_persistence.ticker_lookup_)}'
    )
    display(fitted_persistence.persistence_table_.head(15))

if plt is not None:
    if 'Baseline E (Lag-Reversal)' in fitted_default_models:
        top_rule_df = fitted_lag_reversal.rule_summary_.head(20).copy()
        top_rule_df['abs_weight'] = top_rule_df['effective_weight'].abs()
        plot_rule_df = top_rule_df.sort_values('abs_weight', ascending=True)
        fig_imp, ax_imp = plt.subplots(figsize=(10, 6))
        bar_colors = ['#10b981' if w >= 0 else '#ef4444' for w in plot_rule_df['effective_weight']]
        ax_imp.barh(plot_rule_df['feature'], plot_rule_df['effective_weight'], color=bar_colors, alpha=0.85)
        ax_imp.axvline(0.0, color='#64748b', linewidth=0.9)
        ax_imp.set_xlabel('Effective Signed Weight')
        ax_imp.set_title('Top 20 Baseline E Features by Effective Weight')
        plt.tight_layout()
        if save_results:
            results_dir.mkdir(parents=True, exist_ok=True)
            plt.savefig(results_dir / f'{timestamp}_{experiment_name}_baseline_e_weights.png', dpi=140)
        plt.show()
    elif 'Baseline A (Bin-Snapping)' in fitted_default_models:
        top_bin_df = lookup_table_df.head(20).copy()
        top_bin_df['abs_corr'] = top_bin_df['train_corr'].abs()
        plot_bin_df = top_bin_df.sort_values('abs_corr', ascending=True)
        fig_imp, ax_imp = plt.subplots(figsize=(10, 6))
        bar_colors = ['#2563eb' if c >= 0 else '#ef4444' for c in plot_bin_df['train_corr']]
        ax_imp.barh(plot_bin_df.index, plot_bin_df['train_corr'], color=bar_colors, alpha=0.85)
        ax_imp.axvline(0.0, color='#64748b', linewidth=0.9)
        ax_imp.set_xlabel('Training Target Correlation')
        ax_imp.set_title('Top 20 Baseline A Features by Training Correlation')
        plt.tight_layout()
        if save_results:
            results_dir.mkdir(parents=True, exist_ok=True)
            plt.savefig(results_dir / f'{timestamp}_{experiment_name}_baseline_a_corrs.png', dpi=140)
        plt.show()

Baseline A Learned Lookup Table (top 10 features):


,bin_0,bin_1,bin_2,bin_3,bin_4,train_corr
pca_comp_13,0.499388,0.510597,0.482000,0.492758,0.447926,-0.078539
feature_intelligence_1_ticker_lag1,0.499669,0.516645,0.489300,0.490010,0.465253,-0.059802
feature_charisma_5_ticker_exp_min,0.468958,0.484018,0.497717,0.518101,0.480137,0.058187
pca_comp_7,0.511260,0.486134,0.496865,0.483686,0.464570,-0.058162
feature_charisma_2_ticker_exp_min,0.515830,0.487049,0.480755,0.491246,0.473452,-0.056005
feature_charisma_5_ticker_exp_mean,0.457314,0.483162,0.491937,0.508934,0.481230,0.054032
feature_charisma_5_ticker_roll_mean4,0.457314,0.483162,0.491937,0.508934,0.481230,0.054032
feature_charisma_5_ticker_roll_mean12,0.457314,0.483162,0.491937,0.508934,0.481230,0.054032
pca_comp_15,0.453618,0.485285,0.503512,0.488163,0.493903,0.053711
feature_charisma_5_ticker_exp_max,0.471843,0.480732,0.496585,0.523349,0.474827,0.050821


Baseline E Signal Rules | kept (+1)=45 | inverted (-1)=28


,feature,target_corr,lag_acf,sign_rule,effective_weight
0,pca_comp_1,0.057809,-0.135489,-0.911282,-0.052680
1,feature_strength_71_ticker_wks_change,0.038259,0.296802,0.999999,0.038259
2,row_feat_mean_bin,0.043685,-0.006965,0.776084,0.033903
3,feature_constitution_50_era_diff1,0.030856,-0.535755,-1.000000,-0.030856
4,fourier_cos_k2,0.030854,-0.920537,-1.000000,-0.030854
5,pca_comp_0,0.032969,-0.138065,-0.919619,-0.030319
6,feature_constitution_233_era_diff1,0.028069,-0.426845,-0.999999,-0.028069
7,feature_constitution_287_era_diff8,0.024888,0.259299,0.999994,0.024888
8,feature_constitution_287_era_diff4,0.024728,0.325000,1.000000,0.024728
9,pca_comp_14,0.032626,-0.107731,-0.753006,-0.024568


Baseline P (Persistence) | mode=None | lag=12 | global_mean=0.4927 | tracked_tickers=0


,ticker,persisted_target,obs_count,last_era


In [14]:
for baseline_name, cv_results_df in cv_results_by_baseline.items():
    print(f'Top CV candidates for {baseline_name}:')
    display(cv_results_df.head(10))
    if save_results:
        results_dir.mkdir(parents=True, exist_ok=True)
        baseline_slug = 'baseline_a' if 'Baseline A' in baseline_name else ('baseline_e' if 'Baseline E' in baseline_name else 'persistence')
        cv_results_path = results_dir / f'{timestamp}_{experiment_name}_{baseline_slug}_{search_mode}_cv.csv'
        cv_results_df.to_csv(cv_results_path, index=False)
        print('Saved CV results CSV:', cv_results_path)

Top CV candidates for Baseline A (Bin-Snapping):


,candidate_id,mean_fit_time,std_fit_time,mean_score_time,std_score_time,split0_test_score,mean_test_score,std_test_score,rank_test_score,mean_test_score_pct,...,extractor__top_k,extractor__variance_threshold,extractor__year_horizon_filter,n_bins,prediction_transform,recency_decay,smoothing,top_k,walk_forward_step_eras,walk_forward_window
0,1,0.496362,0.0,1.424868,0.0,-0.027849,-0.027849,0.0,1,-2.78,...,NaN,NaN,"[1, 2, 3]",5,era_rank,0.015708,57.701596,10.0,NaN,expanding
1,2,0.216055,0.0,0.832766,0.0,-0.038616,-0.038616,0.0,2,-3.86,...,NaN,0.02,"[1, 2, 3]",5,None,0.012675,20.634178,10.0,NaN,expanding
2,0,0.318177,0.0,1.167432,0.0,-0.052990,-0.052990,0.0,3,-5.30,...,NaN,NaN,"[4, 5, 6]",3,None,0.002128,61.557147,40.0,20.0,expanding
3,4,1.157614,0.0,1.456349,0.0,-0.058934,-0.058934,0.0,4,-5.89,...,40.0,0.10,"[7, 8, 9]",3,era_rank,0.019819,9.198171,NaN,40.0,expanding
4,3,13.288689,0.0,31.707105,0.0,-0.063598,-0.063598,0.0,5,-6.36,...,10.0,0.05,None,3,None,0.013176,16.801149,10.0,20.0,expanding


Saved CV results CSV: C:\Users\maran\OneDrive\Documents\Git Profile\Data-Projects\numerai\results_numerai_v6.0_all_eras_stride4_baselines_ae_target_bayes\20261125_000357_numerai_v6.0_all_eras_stride4_baselines_ae_baseline_a_bayes_cv.csv
Top CV candidates for Baseline E (Lag-Reversal):


,candidate_id,mean_fit_time,std_fit_time,mean_score_time,std_score_time,split0_test_score,mean_test_score,std_test_score,rank_test_score,mean_test_score_pct,...,extractor__top_k,extractor__variance_threshold,extractor__year_horizon_filter,lag,prediction_transform,recency_decay,sign_mode,top_k,walk_forward_step_eras,walk_forward_window
0,4,3.162368,0.0,0.816448,0.0,-0.037143,-0.037143,0.0,1,-3.71,...,NaN,0.05,None,1,None,0.010469,tanh,20.0,NaN,expanding
1,0,0.516122,0.0,2.431116,0.0,-0.038260,-0.038260,0.0,2,-3.83,...,NaN,NaN,"[4, 5, 6]",1,era_zscore,0.018425,soft_threshold,10.0,10.0,expanding
2,2,0.335035,0.0,1.948666,0.0,-0.054419,-0.054419,0.0,3,-5.44,...,10.0,0.05,"[4, 5, 6]",2,era_zscore,0.002260,hard,10.0,20.0,expanding
3,3,0.066258,0.0,1.290183,0.0,-0.057047,-0.057047,0.0,4,-5.70,...,20.0,0.05,"[1, 2, 3]",3,era_zscore,0.008611,soft_threshold,NaN,10.0,sliding
4,1,1.070713,0.0,2.131422,0.0,-0.085280,-0.085280,0.0,5,-8.53,...,NaN,NaN,"[7, 8, 9]",4,era_zscore,0.016878,tanh,20.0,20.0,sliding


Saved CV results CSV: C:\Users\maran\OneDrive\Documents\Git Profile\Data-Projects\numerai\results_numerai_v6.0_all_eras_stride4_baselines_ae_target_bayes\20261125_000357_numerai_v6.0_all_eras_stride4_baselines_ae_baseline_e_bayes_cv.csv
Top CV candidates for Baseline P (Persistence):


,candidate_id,mean_fit_time,std_fit_time,mean_score_time,std_score_time,split0_test_score,mean_test_score,std_test_score,rank_test_score,mean_test_score_pct,...,worst_round_gbp,bias_correction,era_stride,era_weighting,persistence_mode,prediction_transform,recency_decay,smoothing,walk_forward_step_eras,walk_forward_window
0,0,0.015613,0.0,0.095588,0.0,0.000000,0.000000,0.0,1,0.00,...,0.00,mean_residual,12,equal_era,None,era_rank,0.002184,0.000000,20.0,sliding
1,2,0.048312,0.0,0.668079,0.0,-0.028704,-0.028704,0.0,2,-2.87,...,-10.01,mean_residual,4,equal_era,expanding_mean,None,0.015260,33.524839,40.0,expanding
2,3,0.048292,0.0,0.635867,0.0,-0.030118,-0.030118,0.0,3,-3.01,...,-15.62,None,4,recency_linear,last,None,0.018683,31.948732,NaN,sliding
3,4,0.016665,0.0,0.371851,0.0,-0.041687,-0.041687,0.0,4,-4.17,...,-14.26,None,12,None,expanding_mean,era_rank,0.011474,81.388902,40.0,sliding
4,1,0.048251,0.0,0.690638,0.0,-0.043041,-0.043041,0.0,5,-4.30,...,-13.51,mean_residual,4,equal_era,expanding_mean,None,0.019523,0.911288,40.0,sliding


Saved CV results CSV: C:\Users\maran\OneDrive\Documents\Git Profile\Data-Projects\numerai\results_numerai_v6.0_all_eras_stride4_baselines_ae_target_bayes\20261125_000357_numerai_v6.0_all_eras_stride4_baselines_ae_persistence_bayes_cv.csv


## Error-Direction Envelope Analysis (Per Baseline Model — Best Highlighted)

In [15]:
envelope_dfs = {}
seen_model_families = set()
envelope_variants = []
ordered_envelope_candidates = [best_variant_name] + [
    variant_name for variant_name in comparison_df['variant'].tolist() if variant_name != best_variant_name
]
for variant_name in ordered_envelope_candidates:
    if variant_name not in baseline_results:
        continue
    model_family = variant_name.split(' [')[0]
    if model_family not in seen_model_families:
        seen_model_families.add(model_family)
        envelope_variants.append(variant_name)

for variant_name in envelope_variants:
    is_best_variant = (variant_name == best_variant_name)
    variant_slug = (
        variant_name.lower()
        .replace(' ', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('[', '')
        .replace(']', '')
        .replace('-', '_')
    )
    envelope_save_path = (
        results_dir / f'{timestamp}_{experiment_name}_{variant_slug}_error_envelope.png'
        if save_results
        else None
    )
    envelope_dfs[variant_name] = plot_era_error_envelope(
        y_validation,
        baseline_results[variant_name]['validation_preds'],
        validation_eras,
        title=variant_name,
        band_size=0.15,
        is_best=is_best_variant,
        save_path=envelope_save_path,
    )

envelope_df = envelope_dfs[best_variant_name]
print(f'Best Model Envelope Summary ({best_variant_name}):')
display(envelope_df.head(10))

Best Model Envelope Summary (Baseline P (Persistence) [Tuned]):


,data_points,under_share,over_share,mae,mean_residual,is_interpolated
era,,,,,,
836,6180,0.750000,0.250000,0.159294,-0.018467,False
837,6169,0.750041,0.249959,0.159239,-0.018467,False
838,6163,0.749959,0.250041,0.159301,-0.018467,False
839,6163,0.749959,0.250041,0.159301,-0.018467,False
840,6144,0.750000,0.250000,0.159278,-0.018466,False
841,6130,0.750082,0.249918,0.159216,-0.018466,False
842,6126,0.749918,0.250082,0.159307,-0.018466,False
843,6127,0.749959,0.250041,0.159285,-0.018466,False
844,6143,0.749959,0.250041,0.159301,-0.018466,False


## Error-Bin Waterfall & Financial Quarter Breakdown

In [16]:
val_dates = val_diag_meta['date'] if 'date' in val_diag_meta.columns else None
model_error_bin_comparison_df = compute_model_error_bin_comparison(
    baseline_results,
    y_validation,
    best_variant_name=best_variant_name,
    mode=error_bin_mode,
    pct_bins=error_pct_bins,
    zero_tols=error_zero_tols,
    abs_bins=error_abs_bins,
)
print('Overall Error-Bin Comparison Across Evaluated Models:')
display(model_error_bin_comparison_df)

waterfall_df = compute_error_bin_waterfall(
    y_validation,
    baseline_results[best_variant_name]['validation_preds'],
    validation_eras,
    dates=val_dates,
    mode=error_bin_mode,
    pct_bins=error_pct_bins,
    zero_tols=error_zero_tols,
    abs_bins=error_abs_bins,
)
print(f'Error-Bin Waterfall Summary Table ({best_variant_name}):')
display(waterfall_df)

waterfall_png_path = (
    results_dir / f'{timestamp}_{experiment_name}_error_bin_waterfall.png'
    if save_results
    else None
)
plot_error_bin_waterfall(
    waterfall_df,
    title=best_variant_name,
    save_path=waterfall_png_path,
)


Overall Error-Bin Comparison Across Evaluated Models:


,variant,data_points,extreme_underforecast_pct,underforecast_pct,accurate_pct,overforecast_pct,extreme_overforecast_pct,net_bias_pct,extreme_total_pct,mae
0,Baseline A (Bin-Snapping) [Default],2660746,25.00,0.00,50.00,0.00,25.00,0.00,50.00,0.1506
1,Baseline A (Bin-Snapping) [Walk-Forward],2660746,25.00,0.00,50.00,0.00,25.00,0.00,50.00,0.1502
2,Baseline A (Bin-Snapping) [Tuned],2660746,44.64,2.51,4.87,2.79,45.19,0.84,89.83,0.2989
3,Baseline A (Bin-Snapping) [Tuned Walk-Forward],2660746,45.05,2.53,4.78,2.71,44.93,0.05,89.98,0.2998
4,Baseline E (Lag-Reversal) [Default],2660746,95.22,1.37,0.57,1.26,1.58,-93.76,96.80,0.5051
5,Baseline E (Lag-Reversal) [Walk-Forward],2660746,95.51,1.52,0.51,1.40,1.07,-94.56,96.58,0.5045
6,Baseline E (Lag-Reversal) [Tuned],2660746,96.85,1.46,0.33,0.96,0.40,-96.96,97.25,0.5735
7,Baseline E (Lag-Reversal) [Tuned Walk-Forward],2660746,95.03,1.39,0.52,2.56,0.49,-93.38,95.53,0.4844
8,Baseline P (Persistence) [Default],2660746,27.97,0.00,43.57,0.00,28.45,0.48,56.43,0.1905
9,Baseline P (Persistence) [Walk-Forward],2660746,29.99,0.00,40.10,0.00,29.91,-0.08,59.90,0.2177


Error-Bin Waterfall Summary Table (Baseline P (Persistence) [Tuned]):


,section,period,eras,data_points,extreme_underforecast_pct,underforecast_pct,accurate_pct,overforecast_pct,extreme_overforecast_pct,net_bias_pct,extreme_total_pct,mae
0,Overall,Overall (All Validation),399,2660746,25.0,0.0,50.00,0.0,25.0,0.0,50.00,0.1593
1,Quarterly Avg,Quarterly Avg (31 Qtrs),13,85831,25.0,0.0,50.00,0.0,25.0,0.0,50.00,0.1593
2,Seasonal Avg,Q1 Avg (8 yrs),13,85597,25.0,0.0,50.00,0.0,25.0,0.0,50.00,0.1593
3,Seasonal Avg,Q2 Avg (8 yrs),13,86571,25.0,0.0,50.00,0.0,25.0,0.0,50.00,0.1593
4,Seasonal Avg,Q3 Avg (8 yrs),12,83502,25.0,0.0,50.00,0.0,25.0,0.0,50.00,0.1593
5,Seasonal Avg,Q4 Avg (7 yrs),13,87912,25.0,0.0,50.00,0.0,25.0,0.0,50.00,0.1593
6,Financial Quarter,2019-Q1,13,80002,25.0,0.0,50.00,0.0,25.0,0.0,50.00,0.1593
7,Financial Quarter,2019-Q2,13,79915,25.0,0.0,50.00,0.0,25.0,0.0,50.00,0.1593
8,Financial Quarter,2019-Q3,13,80067,25.0,0.0,50.00,0.0,25.0,0.0,50.00,0.1593
9,Financial Quarter,2019-Q4,13,82594,25.0,0.0,49.99,0.0,25.0,0.0,50.01,0.1593


## AFML Backtest Overfitting, Risk & Numerai Forum #1454 Diagnostics
Summarizes **Era Consistency (`CORR > 0.002`)**, **Lo (2002) autocorrelation-adjusted Sharpe**, **PSR**, **ONC-Clustered $E[K]$ DSR & Šidák FWER $\alpha_K$**, **MinTRL**, **CSCV Probability of Backtest Overfitting (PBO)**, **Feature Neutral Correlation (FNC)**, **Time under Water (TuW)**, **Return Concentration (HHI)**, and **AFML Ch. 15 Strategy Failure Probability**.

In [ ]:
afml_diag_rows = []
for variant_name, res_entry in baseline_results.items():
    base_key = variant_name.split(' [')[0]
    cv_df_ref = cv_results_by_baseline.get(base_key)
    v_trials_mat = cv_df_ref.attrs.get('trials_era_corrs_df') if cv_df_ref is not None else None
    v_trial_srs = (
        cv_df_ref['mean_sharpe'].dropna().to_numpy(dtype=np.float64)
        if (cv_df_ref is not None and 'mean_sharpe' in cv_df_ref.columns and len(cv_df_ref) > 0 and '[Tuned' in variant_name)
        else [res_entry['validation_sharpe']]
    )
    afml_diag_rows.append(
        summarize_afml_backtest_diagnostics(
            res_entry['validation_era_corrs'],
            trial_srs=v_trial_srs,
            trials_matrix=v_trials_mat if '[Tuned' in variant_name else None,
            y_true=y_validation,
            y_pred=res_entry['validation_preds'],
            features_df=X_validation,
            feature_columns=feature_columns,
            eras=validation_eras,
            label=variant_name,
            random_state=random_state,
        )
    )
afml_diagnostics_df = pd.DataFrame(afml_diag_rows)
print('AFML Backtest Risk, Multiple-Testing (ONC-Clustered DSR / PBO) & Numerai Forum #1454 Diagnostics:')
display(afml_diagnostics_df)
if save_results:
    results_dir.mkdir(parents=True, exist_ok=True)
    afml_diagnostics_df.to_csv(results_dir / f'{timestamp}_{experiment_name}_afml_diagnostics.csv', index=False)


## NumerAPI v6.0 Dataset Discovery & True MMC vs BMC Proxy Check
Lists every published `v6.0/` file via `NumerAPI`, checks whether true `meta_model.parquet` exists, and falls back cleanly to a v6 benchmark/example predictions file labeled **BMC proxy (benchmark, NOT true MMC)**.

In [17]:
v6_contrib_info = discover_and_fetch_v6_meta_or_benchmark(
    data_version=data_version,
    data_folder=data_folder,
    workspace_root=workspace_root,
    auto_download=True,
    verbose=True,
)

=== NumerAPI Dataset Discovery (v6.0) ===
All v6.0 files found (14):
  - v6.0/features.json
  - v6.0/live.parquet
  - v6.0/live_benchmark_models.parquet
  - v6.0/live_example_preds.csv
  - v6.0/live_example_preds.parquet
  - v6.0/live_neutralizer.parquet
  - v6.0/train.parquet
  - v6.0/train_benchmark_models.parquet
  - v6.0/train_neutralizer.parquet
  - v6.0/validation.parquet
  - v6.0/validation_benchmark_models.parquet
  - v6.0/validation_example_preds.csv
  - v6.0/validation_example_preds.parquet
  - v6.0/validation_neutralizer.parquet

Filtered (meta, benchmark, neutralizer, example) (10):
  - v6.0/live_benchmark_models.parquet
  - v6.0/live_example_preds.csv
  - v6.0/live_example_preds.parquet
  - v6.0/live_neutralizer.parquet
  - v6.0/train_benchmark_models.parquet
  - v6.0/train_neutralizer.parquet
  - v6.0/validation_benchmark_models.parquet
  - v6.0/validation_example_preds.csv
  - v6.0/validation_example_preds.parquet
  - v6.0/validation_neutralizer.parquet

[MMC Status] No 

## Simulated Numerai Staking P&L Dashboard (£)


In [18]:
val_mmc_df = val_diag_meta.assign(**{TARGET_COL: y_validation.to_numpy(dtype=np.float32)})
staking_basis_label = v6_contrib_info['basis_label']
staking_is_proxy = bool(v6_contrib_info['mmc_is_proxy'])
print(f"Active Staking Basis: {staking_basis_label}")
print(f"  -> Reason: {v6_contrib_info['reason']}")

stake_pnl_by_variant = {}
variant_era_mmc_by_variant = {}
live_scenarios_by_variant = {}
staking_summary_rows = []
for variant_name in comparison_df['variant'].tolist():
    is_best_variant = (variant_name == best_variant_name)
    variant_slug = (
        variant_name.lower()
        .replace(' ', '_')
        .replace('[', '')
        .replace(']', '')
        .replace('-', '_')
    )
    variant_era_corrs = baseline_results[variant_name]['validation_era_corrs']
    variant_era_contrib = compute_per_era_mmc(
        val_mmc_df,
        baseline_results[variant_name]['validation_preds'],
        meta_model_path=v6_contrib_info['meta_model_path'],
        benchmark_path=v6_contrib_info['benchmark_path'],
        is_benchmark_proxy=staking_is_proxy,
        era_col=ERA_COL,
        target_col=TARGET_COL,
    )
    variant_era_mmc_by_variant[variant_name] = variant_era_contrib

    live_scenarios_by_variant[variant_name] = simulate_live_staking_scenarios(
        variant_era_corrs,
        per_era_mmc=variant_era_contrib,
        mmc_is_proxy=staking_is_proxy,
        total_capital=1000.0,
        settlement_rounds=12,
        weekly_concurrent_rounds=12,
        conservative_payout_factor=0.5,
        live_payout_factor=0.5,
        bmc_proxy_haircut=0.5,
        n_sims=500,
        block_size=12,
        random_state=random_state,
        title=variant_name,
        era_dates=val_era_dates,
    )

    title_label = f'[CV-LOCKED WINNER] {variant_name}' if is_best_variant else variant_name
    pnl_save_path = (
        results_dir / f'{timestamp}_{experiment_name}_{variant_slug}_stake_pnl_dashboard.png'
        if save_results
        else None
    )
    haircutted_contrib = (
        pd.Series(variant_era_contrib, dtype=np.float64) * (0.5 if staking_is_proxy else 1.0)
        if variant_era_contrib is not None
        else None
    )
    if haircutted_contrib is not None:
        haircutted_contrib.name = 'bmc_proxy' if staking_is_proxy else 'mmc'
    stake_pnl_by_variant[variant_name] = plot_stake_pnl_dashboard(
        variant_era_corrs,
        per_era_mmc=haircutted_contrib,
        mmc_is_proxy=staking_is_proxy,
        conservative_pnl=live_scenarios_by_variant[variant_name]['conservative_floor']['pnl'],
        title=f'{title_label} [Realistic Live vs Conservative Floor | PF=0.5, R=12]',
        total_capital=1000.0,
        concurrent_rounds=12,
        settlement_rounds=12,
        payout_factor=0.5,
        payout_rule='auto',
        staking_mode='compound',
        era_dates=val_era_dates,
        save_path=pnl_save_path,
    )

    v_pnl = stake_pnl_by_variant[variant_name]
    c_pnl = live_scenarios_by_variant[variant_name]['conservative_floor']['pnl']
    mean_contrib = float(variant_era_contrib.mean()) if variant_era_contrib is not None and len(variant_era_contrib) > 0 else np.nan
    contrib_col_name = 'validation_bmc_proxy' if staking_is_proxy else 'validation_true_mmc'
    staking_summary_rows.append({
        'variant': variant_name,
        'staking_basis': v_pnl['score_basis'],
        'staking_mode': 'One-Wallet (Compounding, PF=0.5, R=12)',
        'validation_corr': round(float(variant_era_corrs.mean()), 4),
        contrib_col_name: round(mean_contrib, 4) if np.isfinite(mean_contrib) else np.nan,
        'start_capital': round(float(v_pnl['start_capital']), 2),
        'end_capital': round(float(v_pnl['end_capital']), 2),
        'total_return_pct': round(float(v_pnl['total_return_pct']), 2),
        'worst_dip_pct': round(float(v_pnl['money_drawdown_pct']), 2),
        'worst_dip_gbp': round(float(v_pnl['max_drawdown_gbp']), 2),
        'worst_round_gbp': round(float(v_pnl['worst_round_gbp']), 2),
    })
    print(
        f"{title_label} | Realistic Live (PF=0.5, 0.5x BMC, R=12): £{v_pnl['end_capital']:,.2f} ({v_pnl['total_return_pct']:+.2f}%, Dip {v_pnl['money_drawdown_pct']:.2f}%) | "
        f"Conservative Floor (CORR only, PF=0.5, R=12): £{c_pnl['end_capital']:,.2f} ({c_pnl['total_return_pct']:+.2f}%, Dip {c_pnl['money_drawdown_pct']:.2f}%) | "
        f"Basis: {v_pnl['score_basis']}"
    )

staking_comparison_df = pd.DataFrame(staking_summary_rows)
live_staking_scenarios_df = build_live_staking_scenarios_df(live_scenarios_by_variant)
comparison_df['staking_basis'] = comparison_df['variant'].map(
    staking_comparison_df.set_index('variant')['staking_basis']
)
if staking_is_proxy:
    comparison_df['validation_bmc_proxy'] = comparison_df['variant'].map(
        staking_comparison_df.set_index('variant')['validation_bmc_proxy']
    )
elif v6_contrib_info['basis_kind'] == 'true_mmc':
    comparison_df['validation_true_mmc'] = comparison_df['variant'].map(
        staking_comparison_df.set_index('variant')['validation_true_mmc']
    )
print('\nDual-Scenario Realistic Live Staking Table (Conservative Floor [CORR only, PF=0.5, R=12] vs Realistic Live [CORR + 0.5x BMC proxy, PF=0.5, R=12]):')
display(live_staking_scenarios_df)
best_stake_pnl = stake_pnl_by_variant[best_variant_name]

Active Staking Basis: BMC proxy (benchmark, NOT true MMC)
  -> Reason: No v6.0 meta_model.parquet published yet, so true MMC / NMMC cannot be computed in backtest. Falling back to validation_benchmark_models.parquet (saved at C:\Users\maran\OneDrive\Documents\Git Profile\Data-Projects\numerai\data_folder\v6.0\validation_benchmark_models.parquet) to compute BMC proxy (benchmark, NOT true MMC).


Baseline A (Bin-Snapping) [Walk-Forward] [Realistic Live | PF=0.5, 0.5x BMC, R=12] | Basis: BMC proxy (benchmark, NOT true MMC) | Start: £1,000 -> End: £1,253.17 (+25.32%) | Worst dip (peak-to-trough): -25.23% (£-396.89) | Worst single round: £-12.88


Baseline P (Persistence) [Walk-Forward] [Realistic Live | PF=0.5, 0.5x BMC, R=12] | Basis: BMC proxy (benchmark, NOT true MMC) | Start: £1,000 -> End: £2,396.55 (+139.66%) | Worst dip (peak-to-trough): -16.77% (£-313.01) | Worst single round: £-30.59


Baseline A (Bin-Snapping) [Tuned Walk-Forward] [Realistic Live | PF=0.5, 0.5x BMC, R=12] | Basis: BMC proxy (benchmark, NOT true MMC) | Start: £1,000 -> End: £1,240.29 (+24.03%) | Worst dip (peak-to-trough): -21.92% (£-340.34) | Worst single round: £-13.36


Baseline A (Bin-Snapping) [Tuned] [Realistic Live | PF=0.5, 0.5x BMC, R=12] | Basis: BMC proxy (benchmark, NOT true MMC) | Start: £1,000 -> End: £1,055.40 (+5.54%) | Worst dip (peak-to-trough): -30.18% (£-456.27) | Worst single round: £-18.26


Baseline A (Bin-Snapping) [Default] [Realistic Live | PF=0.5, 0.5x BMC, R=12] | Basis: BMC proxy (benchmark, NOT true MMC) | Start: £1,000 -> End: £768.89 (-23.11%) | Worst dip (peak-to-trough): -39.30% (£-427.10) | Worst single round: £-12.40


... (4 outputs left out)

Baseline E (Lag-Reversal) [Default] [Realistic Live | PF=0.5, 0.5x BMC, R=12] | Basis: BMC proxy (benchmark, NOT true MMC) | Start: £1,000 -> End: £520.87 (-47.91%) | Worst dip (peak-to-trough): -58.20% (£-609.23) | Worst single round: £-18.32


Baseline E (Lag-Reversal) [Walk-Forward] [Realistic Live | PF=0.5, 0.5x BMC, R=12] | Basis: BMC proxy (benchmark, NOT true MMC) | Start: £1,000 -> End: £486.21 (-51.38%) | Worst dip (peak-to-trough): -58.14% (£-611.19) | Worst single round: £-18.82


Baseline E (Lag-Reversal) [Tuned Walk-Forward] [Realistic Live | PF=0.5, 0.5x BMC, R=12] | Basis: BMC proxy (benchmark, NOT true MMC) | Start: £1,000 -> End: £529.81 (-47.02%) | Worst dip (peak-to-trough): -51.38% (£-558.31) | Worst single round: £-14.67

Dual-Scenario Realistic Live Staking Table (Conservative Floor [CORR only, PF=0.5, R=12] vs Realistic Live [CORR + 0.5x BMC proxy, PF=0.5, R=12]):


,variant,conservative_end_gbp,conservative_return_pct,conservative_worst_dip_pct,conservative_mc_p_profit_pct,realistic_live_end_gbp,realistic_live_return_pct,realistic_live_worst_dip_pct,realistic_live_mc_p_profit_pct,realistic_live_mc_median_gbp,payout_factor,bmc_haircut,concurrent_rounds
0,Baseline A (Bin-Snapping) [Walk-Forward],1209.60,20.96,-3.54,100.0,1253.17,25.32,-25.23,94.2,1311.91,0.5,0.5,12
1,Baseline P (Persistence) [Walk-Forward],1189.40,18.94,-10.53,96.8,2396.55,139.66,-16.77,99.6,2491.23,0.5,0.5,12
2,Baseline A (Bin-Snapping) [Tuned Walk-Forward],1077.19,7.72,-6.08,91.8,1240.29,24.03,-21.92,89.2,1266.79,0.5,0.5,12
3,Baseline A (Bin-Snapping) [Tuned],1019.12,1.91,-8.90,64.0,1055.40,5.54,-30.18,68.2,1097.54,0.5,0.5,12
4,Baseline A (Bin-Snapping) [Default],996.60,-0.34,-10.01,46.2,768.89,-23.11,-39.30,11.4,784.96,0.5,0.5,12
5,Baseline P (Persistence) [Tuned],1000.00,0.00,-0.00,100.0,1000.00,0.00,-0.00,100.0,1000.00,0.5,0.5,12
6,Baseline P (Persistence) [Tuned Walk-Forward],1000.00,0.00,-0.00,100.0,1000.00,0.00,-0.00,100.0,1000.00,0.5,0.5,12
7,Baseline P (Persistence) [Default],957.25,-4.28,-10.39,21.0,930.26,-6.97,-26.62,38.0,944.81,0.5,0.5,12
8,Baseline E (Lag-Reversal) [Tuned],907.39,-9.26,-12.21,6.8,1032.50,3.25,-22.23,63.4,1072.56,0.5,0.5,12
9,Baseline E (Lag-Reversal) [Default],870.45,-12.96,-19.00,1.6,520.87,-47.91,-58.20,0.2,541.33,0.5,0.5,12


## Monte Carlo Block-Bootstrap Risk Analysis (£)


In [19]:
mc_risk_by_variant = {}
explainer_printed = False
staking_is_proxy = bool(v6_contrib_info['mmc_is_proxy'])
for variant_name in comparison_df['variant'].tolist():
    is_best_variant = (variant_name == best_variant_name)
    variant_slug = (
        variant_name.lower()
        .replace(' ', '_')
        .replace('[', '')
        .replace(']', '')
        .replace('-', '_')
    )
    mc_save_path = (
        results_dir / f'{timestamp}_{experiment_name}_{variant_slug}_monte_carlo_pnl.png'
        if save_results
        else None
    )
    title_label = f'[CV-LOCKED WINNER] {variant_name}' if is_best_variant else variant_name
    raw_contrib = variant_era_mmc_by_variant.get(variant_name)
    haircutted_contrib = (
        pd.Series(raw_contrib, dtype=np.float64) * (0.5 if staking_is_proxy else 1.0)
        if raw_contrib is not None
        else None
    )
    if haircutted_contrib is not None:
        haircutted_contrib.name = 'bmc_proxy' if staking_is_proxy else 'mmc'
    v_mc = plot_stake_pnl_monte_carlo(
        baseline_results[variant_name]['validation_era_corrs'],
        per_era_mmc=haircutted_contrib,
        mmc_is_proxy=staking_is_proxy,
        title=f'{title_label} [Realistic Live: PF=0.5, 0.5x BMC, R=12]',
        total_capital=1000.0,
        concurrent_rounds=12,
        settlement_rounds=12,
        payout_factor=0.5,
        payout_rule='auto',
        staking_mode='compound',
        n_sims=500,
        block_size=12,
        random_state=random_state,
        era_dates=val_era_dates,
        save_path=mc_save_path,
    )
    mc_risk_by_variant[variant_name] = v_mc
    c_mc = live_scenarios_by_variant[variant_name]['conservative_floor']['mc']
    print('=' * 100)
    print(f"{title_label} — {v_mc['mc_verdict_text']}")
    if not explainer_printed:
        print()
        print(v_mc['mc_explainer_text'])
        explainer_printed = True
    print()
    print(v_mc['mc_summary_line'])
    print(f"  -> {v_mc['mc_drawdown_explanation_line']}")
    print(
        f"  -> Conservative Floor MC (CORR only, PF=0.5, R=12): P(Profit)={c_mc['mc_prob_profit_pct']:.1f}% | "
        f"Median=£{c_mc['mc_median_final_stake']:,.2f} | 95% Worst Dip={c_mc['mc_p95_money_drawdown_pct']:.2f}%"
    )
    print('=' * 100)

Baseline A (Bin-Snapping) [Walk-Forward] — VERDICT: Median path MAKES money. Median finish £1,148.50 from £1,000 start (+14.8%). 90.0% of 500 simulated paths finish above £1,000 (10.0% lose money).

How to read Drawdown vs Final Balance (Plain English):
  • Drawdown is the biggest peak-to-trough dip DURING the journey — NOT your final profit or loss.
  • Worked example (Winning path with a drawdown): Start £1,000 -> rises to £1,200 peak -> dips to £1,100 (-8.3% drawdown from peak) -> recovers to £1,300. You still finish +£300 ABOVE £1,000 despite the dip.
  • Worked example (Losing path with a small drawdown): Start £1,000 -> drifts down to £955 (-4.5% dip) and ends at £984.81. A small drawdown still finishes BELOW £1,000 if it never recovers.
  • Median (50th) vs 95th percentile are DIFFERENT paths: Median = typical path; 95th = lucky top 5% of paths.
  • One-Wallet MC: Simulates ONE continuous wallet from start to finish over the full validation period with no resets.
  • Within-Wind

Baseline P (Persistence) [Walk-Forward] — VERDICT: Median path MAKES money. Median finish £1,902.23 from £1,000 start (+90.2%). 100.0% of 500 simulated paths finish above £1,000 (0.0% lose money).

Baseline P (Persistence) [Walk-Forward] [One-Wallet MC | BMC proxy (benchmark, NOT true MMC) | Compounding] | P(Profit): 100.0% (0.0% lose) | Median (50th): £1,902.23 (+90.2%) | 5th Pct: £1,432.27 | 95th Pct: £2,555.37 | 95% Worst Dip: -12.21% | P(Ruin): 0.0%
  -> What drawdown means: 95% Worst Dip -12.21% means in 95% of paths the worst fall from peak was 12.21% or less — this is NOT the final loss.


Baseline A (Bin-Snapping) [Tuned Walk-Forward] — VERDICT: Median path MAKES money. Median finish £1,169.06 from £1,000 start (+16.9%). 89.2% of 500 simulated paths finish above £1,000 (10.8% lose money).

Baseline A (Bin-Snapping) [Tuned Walk-Forward] [One-Wallet MC | BMC proxy (benchmark, NOT true MMC) | Compounding] | P(Profit): 89.2% (10.8% lose) | Median (50th): £1,169.06 (+16.9%) | 5th Pct: £966.85 | 95th Pct: £1,433.50 | 95% Worst Dip: -14.04% | P(Ruin): 0.0%
  -> What drawdown means: 95% Worst Dip -14.04% means in 95% of paths the worst fall from peak was 14.04% or less — this is NOT the final loss.


Baseline A (Bin-Snapping) [Tuned] — VERDICT: Median path MAKES money. Median finish £1,068.34 from £1,000 start (+6.8%). 70.4% of 500 simulated paths finish above £1,000 (29.6% lose money).

Baseline A (Bin-Snapping) [Tuned] [One-Wallet MC | BMC proxy (benchmark, NOT true MMC) | Compounding] | P(Profit): 70.4% (29.6% lose) | Median (50th): £1,068.34 (+6.8%) | 5th Pct: £881.75 | 95th Pct: £1,328.63 | 95% Worst Dip: -19.56% | P(Ruin): 0.0%
  -> What drawdown means: 95% Worst Dip -19.56% means in 95% of paths the worst fall from peak was 19.56% or less — this is NOT the final loss.


Baseline A (Bin-Snapping) [Default] — VERDICT: Median path LOSES money. Median finish £840.04 from £1,000 start (-16.0%). Only 9.2% of 500 simulated paths finish above £1,000 (90.8% lose money). This model is NOT profitable on average in simulation.

Baseline A (Bin-Snapping) [Default] [One-Wallet MC | BMC proxy (benchmark, NOT true MMC) | Compounding] | P(Profit): 9.2% (90.8% lose) | Median (50th): £840.04 (-16.0%) | 5th Pct: £684.02 | 95th Pct: £1,037.93 | 95% Worst Dip: -33.48% | P(Ruin): 0.0%
  -> What drawdown means: 95% Worst Dip -33.48% means in 95% of paths the worst fall from peak was 33.48% or less — this is NOT the final loss.


[CV-LOCKED WINNER] Baseline P (Persistence) [Tuned] — VERDICT: Flat / NO active signal. Median finish £1,000.00 from £1,000 start (+0.0%). 0.0% of 500 simulated paths make money (model outputs a constant prediction).

[CV-LOCKED WINNER] Baseline P (Persistence) [Tuned] [One-Wallet MC | BMC proxy (benchmark, NOT true MMC) | Compounding] | P(Profit): 0.0% (100.0% lose) | Median (50th): £1,000.00 (+0.0%) | 5th Pct: £1,000.00 | 95th Pct: £1,000.00 | 95% Worst Dip: 0.00% | P(Ruin): 0.0%
  -> What drawdown means: 95% Worst Dip 0.00% means in 95% of paths the worst fall from peak was 0.00% or less — this is NOT the final loss.


... (2 outputs left out)

Baseline E (Lag-Reversal) [Tuned] — VERDICT: Median path MAKES money. Median finish £1,097.52 from £1,000 start (+9.8%). 77.0% of 500 simulated paths finish above £1,000 (23.0% lose money).

Baseline E (Lag-Reversal) [Tuned] [One-Wallet MC | BMC proxy (benchmark, NOT true MMC) | Compounding] | P(Profit): 77.0% (23.0% lose) | Median (50th): £1,097.52 (+9.8%) | 5th Pct: £899.25 | 95th Pct: £1,341.44 | 95% Worst Dip: -17.91% | P(Ruin): 0.0%
  -> What drawdown means: 95% Worst Dip -17.91% means in 95% of paths the worst fall from peak was 17.91% or less — this is NOT the final loss.


Baseline E (Lag-Reversal) [Default] — VERDICT: Median path LOSES money. Median finish £668.29 from £1,000 start (-33.2%). Only 0.2% of 500 simulated paths finish above £1,000 (99.8% lose money). This model is NOT profitable on average in simulation.

Baseline E (Lag-Reversal) [Default] [One-Wallet MC | BMC proxy (benchmark, NOT true MMC) | Compounding] | P(Profit): 0.2% (99.8% lose) | Median (50th): £668.29 (-33.2%) | 5th Pct: £521.25 | 95th Pct: £852.03 | 95% Worst Dip: -49.23% | P(Ruin): 0.0%
  -> What drawdown means: 95% Worst Dip -49.23% means in 95% of paths the worst fall from peak was 49.23% or less — this is NOT the final loss.


Baseline E (Lag-Reversal) [Walk-Forward] — VERDICT: Median path LOSES money. Median finish £639.54 from £1,000 start (-36.0%). Only 0.0% of 500 simulated paths finish above £1,000 (100.0% lose money). This model is NOT profitable on average in simulation.

Baseline E (Lag-Reversal) [Walk-Forward] [One-Wallet MC | BMC proxy (benchmark, NOT true MMC) | Compounding] | P(Profit): 0.0% (100.0% lose) | Median (50th): £639.54 (-36.0%) | 5th Pct: £497.59 | 95th Pct: £803.79 | 95% Worst Dip: -50.95% | P(Ruin): 0.0%
  -> What drawdown means: 95% Worst Dip -50.95% means in 95% of paths the worst fall from peak was 50.95% or less — this is NOT the final loss.


Baseline E (Lag-Reversal) [Tuned Walk-Forward] — VERDICT: Median path LOSES money. Median finish £677.35 from £1,000 start (-32.3%). Only 0.0% of 500 simulated paths finish above £1,000 (100.0% lose money). This model is NOT profitable on average in simulation.

Baseline E (Lag-Reversal) [Tuned Walk-Forward] [One-Wallet MC | BMC proxy (benchmark, NOT true MMC) | Compounding] | P(Profit): 0.0% (100.0% lose) | Median (50th): £677.35 (-32.3%) | 5th Pct: £555.91 | 95th Pct: £838.77 | 95% Worst Dip: -45.21% | P(Ruin): 0.0%
  -> What drawdown means: 95% Worst Dip -45.21% means in 95% of paths the worst fall from peak was 45.21% or less — this is NOT the final loss.


## Optional Community Companion: `numereval` (Council of Elders)
Runs `councilofelders/numereval` alongside official `numerai-tools` metrics when installed locally (skips gracefully on Kaggle if unavailable). **Note:** `numereval` uses classic Spearman diagnostics and is not the official v6.0 Unity `NCORR` / `NMMC` / staking calculation.

In [20]:
numereval_summary_df = run_numereval_diagnostics(
    y_validation,
    baseline_results[best_variant_name]['validation_preds'],
    validation_eras,
    variant_name=best_variant_name,
    plot=False,
    verbose=True,
)
if numereval_summary_df is not None:
    display(numereval_summary_df)

[numereval companion — Baseline P (Persistence) [Tuned]] Community Spearman diagnostics (informational alongside official numerai_tools v6.0 metrics; NOT official v6.0 NCORR/NMMC):


,metrics
mean,-0.0010
std,0.0086
sharpe,-0.1192
max_drawdown,-0.4863
